In [1]:
### Preamples ###
import numpy as np
import matplotlib.pyplot as plt
import random
from numba import njit

random.seed(13)

In [ ]:
### Simulation ###
@njit
def heston_returns(V0, N, mu, theta, kappa, sigma_V, rho, h, n_sub):
    
    dt = h / n_sub
    sqrt_dt = dt ** 0.5
    c = (1 - rho ** 2) ** 0.5 
    
    returns = np.empty(N)
  
    Vt = V0
    log_ret = 0
    
    for n in range(N):
        log_ret = 0
        for _ in range(n_sub):
            z1 = np.random.normal(0, 1)
            z2 = np.random.normal(0, 1)
            
            dB1 = sqrt_dt * z1
            dB2 = sqrt_dt * (rho * z1 + c * z2) 
            
            Vt_pos = Vt if Vt > 0 else 0 # ensure variance is non-negative for the price update
            
            log_ret += (mu - 0.5 * Vt) * dt + Vt_pos ** 0.5 * dB1 # log return increment
            
            Vt = Vt_pos + kappa * (theta - Vt_pos)* dt + sigma_V * Vt_pos ** 0.5 * dB2  # variance process
            
        returns[n] = log_ret
        
    return returns 

: 

In [ ]:
## General Lag-M moments for kappa estimation
def lagm_cov_est(ret, m):
    
    N = len(ret)
    Ybar = np.mean(ret)
    
    cov = np.sum((ret[:N-m] - Ybar) * (ret[m:] - Ybar)) / (N - m)
    return cov

def estimate_kappa(ret, h , lag_M = 10):
    N = len(ret)
    
    lag1 = lagm_cov_est(ret, 1)
    
    kappa_sum = 0
    
    for m in range(2, lag_M + 1):
        lagm = lagm_cov_est(ret, m)
        
        if lagm == 0 or np.sign(lag1) != np.sign(lagm):
            continue
        
        ratio = lag1 / lagm
        if ratio <= 0 or not np.isfinite(np.log(ratio)):
            continue
        
        k_est = np.log(ratio) / ((m-1) * h)
        if not np.isfinite(k_est) or k_est <= 0:
            continue
        kappa_sum += k_est
        
    kappa = kappa_sum / (lag_M - 1)
    return kappa

def h_tilde(kappa, h):
    "h_tidle = (1 - exp(-kappa * h)) / kappa"
    return (1 - np.exp(-kappa * h)) / kappa

: 

In [ ]:
def moments(mu, theta, kappa, sigma_V, rho, h):
    """Returns first five moments of the log returns of the Heston model."""
    ht = h_tilde(kappa, h)
    ek = np.exp(-kappa * h)
    
    # First moment (mean)
    
    mean = h * (mu  - 0.5 * theta)
    
    # Second moment (variance)
    
    var = theta * (h - ht) * ( (sigma_V**2)/ (4 * (kappa ** 2)) 
                              - (rho * sigma_V) / kappa) + theta * h
    
    # Third moment (lag-1 autocovariance), follows from Theorem 2.3
    
    lag1 = (((ht ** 2) * theta * (sigma_V)) / 2 ) * ( (sigma_V) / (4 * kappa) - rho)
    
    # Fourth moment (lag-2 autocovariance)
    
    lag2 = ek * lag1
    
    
    # Fifth moment (Cross variance bewteen squared retursn and future returns),
    
    
    term1 = ((theta * (sigma_V ** 4) / ( 8 * kappa **3))) * ht * (h * ek - ht)
    
    term2 = ht**2 * ((theta * sigma_V ** 2) / (4 * kappa) * mu * h
                     - (theta ** 2 * sigma_V ** 2) / (8 * kappa) * h
                     - (theta * sigma_V ** 2 ) / (4 * kappa))
    
    term3  = (-rho * sigma_V / 2 * ht) * (
        (3 * sigma_V**2 / (2 * kappa**2) - 2 * rho * sigma_V / kappa)
        * theta * (h * ek - ht)
        + (2 * mu * theta - theta**2) * h * ht
)
    
    cross = term1 + term2 - term3
    
    return mean, var, lag1, lag2, cross

: 

In [ ]:
def estimate_moments(ret):
    n = len(ret)
    Ybar = np.mean(ret)
    mean = Ybar
    var = np.var(ret, ddof = 1)
    lag1 = lagm_cov_est(ret, 1) 
    lag2 = lagm_cov_est(ret, 2)

    y2 = ret[:-1]**2
    y2d = y2 - np.mean(y2)
    yn1d = ret[1:] - np.mean(ret[1:])
    cross = np.mean(y2d * yn1d)
    
    return np.array([mean, var, lag1, lag2, cross])

def estimate_heston_from_returns(ret, h):
    N = len(ret)
    mean_e, var_e, lag1_e, lag2_e, cross_e = estimate_moments(ret)
    
    kappa = estimate_kappa(ret, h)
    print( f"  κ̂    = {kappa:.4f}")
    
    ht = h_tilde(kappa, h)
    
    ek = np.exp(-kappa * h)
    
    dh = h * ek - ht
    
    theta = var_e / h  - (2* (h-ht) / (h * kappa * ht ** 2)) * lag1_e 
    print(f"  θ̂    = {theta:.4f}")
    
    mu = mean_e / h + theta / 2
    print(f"  μ̂    = {mu:.4f}")
    
    num   = (4*kappa*mean_e
             + (8*dh/(theta*ht**3))*lag1_e
             - 2*kappa*cross_e/lag1_e)
    den   = (theta*ht**2/(2*lag1_e) - dh/(kappa*ht))

    sigma_sq = num / den
    sigma_V = sigma_sq**0.5
    print(f"  σ̂_V  = {sigma_V:.6f}")

    rho = np.clip(sigma_V/(4*kappa) - (2/(theta*sigma_V*ht**2))*lag1_e,
                  -0.999, 0.999)
    print(f"  ρ̂    = {rho:.6f}")

    return {'mu': mu, 'theta': theta, 'kappa': kappa,
            'sigma_V': sigma_V, 'rho': rho}
    
    
    

: 

In [ ]:
TRUE = dict(mu=0.125, kappa=0.1, theta=0.25, sigma_V=0.1, rho=-0.7)
V0 = TRUE['theta']    # start in stationarity
h = 1.0              # annual observation interval
N = int(400_000)          # number of observations
n_sub = int(20)              # sub-steps per interval — critical for stability

# ── Simulate ──────────────────────────────────────────────────────────────────
ret = heston_returns(V0, N,                        # no S0 — returns only
                     TRUE['mu'], TRUE['theta'], TRUE['kappa'],
                     TRUE['sigma_V'], TRUE['rho'], h, n_sub)

print(f"Returns shape : {ret.shape}")
print(f"Any NaN       : {np.isnan(ret).any()}")
print(f"Mean return   : {ret.mean():.6f}")
print(f"Std return    : {ret.std():.6f}")

# ── Moments comparison ────────────────────────────────────────────────────────
# Empirical — from simulated returns
samp = estimate_moments(ret)                       # uses lagm_cov_est internally

# Theoretical — from true parameters at h=1
mean_t, var_t, lag1_t, lag2_t, cross_t = moments(**TRUE, h=h)
anal = np.array([mean_t, var_t, lag1_t, lag2_t, cross_t])

names = ['Mean', 'Variance', 'Lag-1 Autocov', 'Lag-2 Autocov', 'Cross-Cov']
print(f"\n{'Moment':<20s}  {'Theoretical':>14s}  {'Empirical':>14s}  {'Rel. Error':>10s}")
print("-" * 68)
for name, a, s in zip(names, anal, samp):
    rel = abs(s-a)/abs(a)*100 if a != 0 else float('nan')
    print(f"  {name:<18s}  {a:+14.6e}  {s:+14.6e}  {rel:>9.1f}%")

# ── Parameter estimation ──────────────────────────────────────────────────────
print("\nEstimating parameters …")
est = estimate_heston_from_returns(ret, h)         # returns dict of params

print(f"\n{'Parameter':<12s}  {'True':>10s}  {'Estimated':>10s}  {'Abs Error':>10s}")
print("-" * 48)
for key in ['mu', 'theta', 'kappa', 'sigma_V', 'rho']:
    t = TRUE[key]; e = est[key]
    print(f"  {key:<10s}  {t:10.4f}  {e:10.4f}  {abs(t-e):10.4f}")

: 

In [ ]:
# Estimate Sigma empirically to construct confidence intervals

# HAC (heteroskedasticity and autocorrelation consistent) estimator
def newey_west(z, lag_max):
    
    N, k = z.shape
    Sigma = (z.T @ z) / N # lag-0 term
    
    for lag in range(1, lag_max + 1):
        w = 1 - lag / (lag_max + 1)
        Gamma = (z[lag:].T @ z[:-lag]) / N
        Sigma += w * (Gamma + Gamma.T)
        
    return Sigma

def moment_conditions(ret, params):
    mu, theta, kappa, sigma_V, rho = params
    N = len(ret)
    Ybar = np.mean(ret)
    Y2bar = np.mean(ret ** 2)
    
    # Moments per obs
    z = np.zeros((N, 5))
    z[:, 0] = ret - Ybar                                    # mean
    z[:, 1] = (ret - Ybar) ** 2 - np.var(ret, ddof = 1)     # var
    cov1 = lagm_cov_est(ret, 1)
    z[:-1, 2] = (ret[:-1] - Ybar) * (ret[1:] - Ybar) - cov1     # lag1

    cov2 = lagm_cov_est(ret, 2)
    z[:-2, 3] = (ret[:-2] - Ybar) * (ret[2:] - Ybar) - cov2     # lag2

    y2    = ret[:-1]**2
    y2bar = np.mean(y2)
    cross = np.mean((y2 - y2bar) * (ret[1:] - np.mean(ret[1:])))
    z[:-1, 4] = (y2 - y2bar) * (ret[1:] - Ybar) - cross         # cross
    return z

: 

In [ ]:
def estimate_from_moments(samp, h):
    mean_e, var_e, lag1_e, lag2_e, cross_e = samp

    if lag1_e == 0 or np.sign(lag1_e) != np.sign(lag2_e):
        raise ValueError(f"Invalid lag ratio.")
    ratio = lag2_e / lag1_e
    if not (0 < ratio < 1):
        raise ValueError(f"Ratio {ratio:.4f} not in (0,1).")

    kappa   = -np.log(ratio) / h
    ht      = (1 - np.exp(-kappa*h)) / kappa
    ek      = np.exp(-kappa*h)
    dh      = h*ek - ht
    theta   = var_e/h - (2*(h-ht)/(h*kappa*ht**2))*lag1_e

    if not np.isfinite(theta) or theta <= 0:
        raise ValueError(f"Invalid theta = {theta:.4e}")

    mu      = mean_e/h + theta/2
    num     = (4*kappa*mean_e
               + (8*dh/(theta*ht**3))*lag1_e
               - 2*kappa*cross_e/lag1_e)
    den     = theta*ht**2/(2*lag1_e) - dh/(kappa*ht)

    if abs(den) < 1e-14:
        raise ValueError("Denominator zero.")

    sigma_sq = num / den


    min_sigma_sq = 1e-10 # clip sigma to ensure Jacobian is well-defined
    if sigma_sq <= 0:
        sigma_sq = min_sigma_sq   # boundary value — derivative still defined

    sigma_V = sigma_sq**0.5
    rho     = np.clip(
        sigma_V/(4*kappa) - (2/(theta*sigma_V*ht**2))*lag1_e,
        -0.999, 0.999
    )

    return {'kappa': kappa, 'theta': theta, 'sigma_V': sigma_V,
            'mu': mu, 'rho': rho}

: 

In [ ]:
def jacobian(samp, est, h, max_entry=1e4):
    """
    Numerical Jacobian with entry capping to prevent explosion
    from near-zero lag estimates in the denominator.
    """
    keys = ['kappa', 'theta', 'sigma_V', 'mu', 'rho']
    k    = len(samp)
    Jg   = np.zeros((5, k))

    for j in range(k):
        eps        = max(abs(samp[j]) * 1e-4, 1e-8)
        samp_up    = samp.copy(); samp_up[j] += eps
        samp_dn    = samp.copy(); samp_dn[j] -= eps

        try:
            est_up = estimate_from_moments(samp_up, h)
            est_dn = estimate_from_moments(samp_dn, h)
            for i, key in enumerate(keys):
                deriv      = (est_up[key] - est_dn[key]) / (2*eps)
                Jg[i, j]   = np.clip(deriv, -max_entry, max_entry)
        except Exception:
            try:
                est_up   = estimate_from_moments(samp_up, h)
                est_base = estimate_from_moments(samp, h)
                for i, key in enumerate(keys):
                    deriv    = (est_up[key] - est_base[key]) / eps
                    Jg[i, j] = np.clip(deriv, -max_entry, max_entry)
            except Exception as ex:
                print(f"  Jacobian column {j} failed: {ex}")

    return Jg

def confidence_intervals(ret, est, h, alpha=0.05):
    from scipy.stats import norm

    N    = len(ret)
    samp = estimate_moments(ret)

    params = [est['mu'], est['theta'], est['kappa'],
              est['sigma_V'], est['rho']]
    z      = moment_conditions(ret, params)
    lag_nw = int(4 * (N/100)**(2/9))
    Sigma  = newey_west(z, lag_nw)
    Jg     = jacobian(samp, est, h)
    V      = Jg @ Sigma @ Jg.T
    diag_V = np.diag(V)

    # Raise if any diagonal entry is zero or negative
    bad_keys = [['kappa','theta','sigma_V','mu','rho'][i]
                for i in range(5) if diag_V[i] <= 0]
    if bad_keys:
        raise ValueError(f"Degenerate Jacobian for {bad_keys} — skipping replication.")

    se     = np.sqrt(np.abs(diag_V) / N)
    z_crit = norm.ppf(1 - alpha/2)

    keys    = ['kappa', 'theta', 'sigma_V', 'mu', 'rho']
    results = {}
    for i, key in enumerate(keys):
        e  = est[key]
        s  = se[i]
        lo = e - z_crit * s
        hi = e + z_crit * s
        results[key] = (e, lo, hi, s)

    return results

: 

In [ ]:
# Run
est = estimate_heston_from_returns(ret, h)
ci = confidence_intervals(ret, est, h)

: 

In [ ]:
def replicated_ci(TRUE, V0, h, n_sub, n_rep=50, N=int(400_000)):
    """
    Runs n_rep independent Heston simulations of length N,
    estimates parameters from each, and reports:
        mean ± std  across replications
        
    Parameters
    ----------
    TRUE  : dict of true parameters
    V0    : initial variance (= TRUE['theta'] for stationarity)
    h     : observation interval
    n_sub : sub-steps per interval
    n_rep : number of independent replications
    N     : observations per replication
    """
    results = {key: [] for key in ['kappa', 'theta', 'sigma_V', 'mu', 'rho']}
    failed  = 0

    print(f"Running {n_rep} replications × {N:,} observations …")
    print(f"(each replication takes ~10 seconds with numba)\n")

    for r in range(n_rep):

        # Simulate one independent path
        ret = heston_returns(V0, N,
                             TRUE['mu'], TRUE['theta'], TRUE['kappa'],
                             TRUE['sigma_V'], TRUE['rho'],
                             h, n_sub)

        # Estimate parameters
        try:
            est = estimate_heston_from_returns(ret, h)
            for key in results:
                results[key].append(est[key])
            if (r + 1) % 10 == 0:
                print(f"  Replication {r+1:>3d} / {n_rep} complete")
        except Exception as ex:
            print(f"  Replication {r+1:>3d} failed: {ex}")
            failed += 1

    print(f"\n  Completed: {n_rep - failed}/{n_rep}  ({failed} failed)\n")

    # Print results in format: mean ± std
    print(f"{'Parameter':<12s}  {'True':>8s}  {'Mean':>10s}  "
          f"{'Std':>10s}  {'Bias':>10s}  {'In 2σ?':>8s}")
    print("-" * 60)

    for key in ['mu', 'theta', 'kappa', 'sigma_V', 'rho']:
        vals = np.array(results[key])
        mean = vals.mean()
        std  = vals.std()
        bias = mean - TRUE[key]
        # Check if true value is within 2 std of mean (≈ 95% band)
        within = '✓' if abs(bias) <= 2*std else '✗'
        print(f"  {key:<10s}  {TRUE[key]:8.4f}  {mean:10.4f}  "
              f"{std:10.4f}  {bias:10.4f}  {within:>8s}")

    return results


# ── Run ───────────────────────────────────────────────────────────────────────
TRUE  = dict(mu=0.125, kappa=0.1, theta=0.25, sigma_V=0.1, rho=-0.7)
V0    = TRUE['theta']
h     = 1.0
n_sub = int(20)


results = replicated_ci(TRUE, V0, h, n_sub, n_rep=100, N=int(400_000))

Running 100 replications × 400,000 observations …
(each replication takes ~10 seconds with numba)



KeyboardInterrupt: 

: 

: 

In [ ]:
def replicated_ci(TRUE, V0, h, n_sub, n_rep=50, N=int(400_000)):
    keys    = ['mu', 'theta', 'kappa', 'sigma_V', 'rho']
    results = {key: [] for key in keys}
    covered = {key: 0  for key in keys}
    failed  = 0

    for r in range(n_rep):
        ret = heston_returns(V0, N, TRUE['mu'], TRUE['theta'], TRUE['kappa'],
                             TRUE['sigma_V'], TRUE['rho'], h, n_sub)
        try:
            est = estimate_heston_from_returns(ret, h)
            ci  = confidence_intervals(ret, est, h)
            for key in keys:
                results[key].append(est[key])
                e, lo, hi, s = ci[key]
                if lo <= TRUE[key] <= hi:
                    covered[key] += 1
        except Exception as ex:
            failed += 1
            print(f"  Replication {r+1:>3d} failed: {ex}")

    valid = n_rep - failed
    print(f"\n  Completed: {valid}/{n_rep}  ({failed} failed)\n")

    if valid == 0:
        print("  All replications failed — cannot compute statistics.")
        return results, failed

    print(f"{'Parameter':<12s}  {'True':>8s}  {'Mean':>10s}  "
          f"{'Std':>10s}  {'Bias':>10s}  {'Coverage':>10s}")
    print("-" * 65)

    for key in keys:
        vals     = np.array(results[key])
        mean     = vals.mean()
        std      = vals.std()
        bias     = mean - TRUE[key]
        cov_rate = covered[key] / valid * 100
        print(f"  {key:<10s}  {TRUE[key]:8.4f}  {mean:10.4f}  "
              f"{std:10.4f}  {bias:10.4f}  {cov_rate:>9.1f}%")

    return results, failed

: 

In [ ]:
def replicated_ci(TRUE, V0, h, n_sub, n_rep=50, N=int(400_000)):
    
    results = {key: [] for key in ['kappa', 'theta', 'sigma_V', 'mu', 'rho']}
    covered = {key: 0 for key in ['kappa', 'theta', 'sigma_V', 'mu', 'rho']}  # NEW
    failed  = 0

    for r in range(n_rep):
        ret = heston_returns(V0, N, TRUE['mu'], TRUE['theta'], TRUE['kappa'],
                             TRUE['sigma_V'], TRUE['rho'], h, n_sub)
        try:
            est = estimate_heston_from_returns(ret, h)
            ci  = confidence_intervals(ret, est, h)          # returns (e, lo, hi, se)
            for key in results:
                results[key].append(est[key])
                e, lo, hi, s = ci[key]
                if lo <= TRUE[key] <= hi:                    # NEW
                    covered[key] += 1                        # NEW
        except Exception as ex:
            failed += 1

    valid = n_rep - failed

    # Print as before, but add coverage column
    print(f"{'Parameter':<12s}  {'True':>8s}  {'Mean':>10s}  "
          f"{'Std':>10s}  {'Bias':>10s}  {'Coverage':>10s}")
    print("-" * 65)

    for key in ['mu', 'theta', 'kappa', 'sigma_V', 'rho']:
        vals     = np.array(results[key])
        mean     = vals.mean()
        std      = vals.std()
        bias     = mean - TRUE[key]
        cov_rate = covered[key] / valid * 100               # NEW
        print(f"  {key:<10s}  {TRUE[key]:8.4f}  {mean:10.4f}  "
              f"{std:10.4f}  {bias:10.4f}  {cov_rate:>9.1f}%")

    return results, failed


failure_counts = []
for meta in range(10):
    results, failures = replicated_ci(TRUE, V0, h, n_sub, n_rep=100, N=int(400_000))
    failure_counts.append(failures)
    
print(f"Failure counts across meta-runs: {failure_counts}")
print(f"Mean failures: {np.mean(failure_counts):.1f}")
print(f"Std failures:  {np.std(failure_counts):.1f}")


: 

In [ ]:
def estimate_from_moments(samp, h, extra_lags=None):
    mean_e, var_e, lag1_e, lag2_e, cross_e = samp

    # --- kappa: multi-lag if available, single ratio as fallback ---
    if extra_lags is not None and len(extra_lags) > 0:
        kappa_sum = 0
        count     = 0
        for m, lagm in enumerate(extra_lags, start=2):
            if lagm == 0 or np.sign(lag1_e) != np.sign(lagm):
                continue
            ratio = lag1_e / lagm
            if ratio <= 0 or not np.isfinite(np.log(ratio)):
                continue
            k_est = np.log(ratio) / ((m - 1) * h)
            if not np.isfinite(k_est) or k_est <= 0:
                continue
            kappa_sum += k_est
            count     += 1
        if count == 0:
            raise ValueError("No valid lag ratios found for kappa.")
        kappa = kappa_sum / count
    else:
        if lag1_e == 0 or np.sign(lag1_e) != np.sign(lag2_e):
            raise ValueError("Invalid lag ratio.")
        ratio = lag2_e / lag1_e
        if not (0 < ratio < 1):
            raise ValueError(f"Ratio {ratio:.4f} not in (0,1).")
        kappa = -np.log(ratio) / h

    ht      = (1 - np.exp(-kappa * h)) / kappa
    ek      = np.exp(-kappa * h)
    dh      = h * ek - ht
    theta   = var_e/h - (2*(h - ht) / (h * kappa * ht**2)) * lag1_e
    if not np.isfinite(theta) or theta <= 0:
        raise ValueError(f"Invalid theta = {theta:.4e}")
    mu      = mean_e/h + theta/2
    num     = (4*kappa*mean_e
               + (8*dh/(theta*ht**3))*lag1_e
               - 2*kappa*cross_e/lag1_e)
    den     = theta*ht**2/(2*lag1_e) - dh/(kappa*ht)
    if abs(den) < 1e-14:
        raise ValueError("Denominator zero.")
    sigma_sq = max(num/den, 1e-10)
    sigma_V  = sigma_sq**0.5
    rho      = np.clip(
        sigma_V/(4*kappa) - (2/(theta*sigma_V*ht**2))*lag1_e,
        -0.999, 0.999
    )
    return {'kappa': kappa, 'theta': theta, 'sigma_V': sigma_V,
            'mu': mu, 'rho': rho}

: 

In [ ]:


def estimate_moments_extended(ret, lag_M=10):
    samp       = estimate_moments(ret)
    extra_lags = np.array([lagm_cov_est(ret, m) for m in range(2, lag_M + 1)])
    return samp, extra_lags


def jacobian(samp, est, h, extra_lags=None, max_entry=1e4):
    keys = ['kappa', 'theta', 'sigma_V', 'mu', 'rho']
    Jg   = np.zeros((5, 5))
    for j in range(5):
        eps     = max(abs(samp[j]) * 1e-4, 1e-8)
        samp_up = samp.copy(); samp_up[j] += eps
        samp_dn = samp.copy(); samp_dn[j] -= eps
        try:
            est_up = estimate_from_moments(samp_up, h, extra_lags)
            est_dn = estimate_from_moments(samp_dn, h, extra_lags)
            for i, key in enumerate(keys):
                deriv     = (est_up[key] - est_dn[key]) / (2*eps)
                Jg[i, j]  = np.clip(deriv, -max_entry, max_entry)
        except Exception:
            try:
                est_up   = estimate_from_moments(samp_up, h, extra_lags)
                est_base = estimate_from_moments(samp,    h, extra_lags)
                for i, key in enumerate(keys):
                    deriv     = (est_up[key] - est_base[key]) / eps
                    Jg[i, j]  = np.clip(deriv, -max_entry, max_entry)
            except Exception as ex:
                print(f"  Jacobian column {j} failed: {ex}")
    return Jg


def confidence_intervals(ret, est, h, alpha=0.05, lag_M=10):
    from scipy.stats import norm
    N                = len(ret)
    samp, extra_lags = estimate_moments_extended(ret, lag_M)
    params = [est['mu'], est['theta'], est['kappa'], est['sigma_V'], est['rho']]
    z      = moment_conditions(ret, params)
    lag_nw = int(4 * (N/100)**(2/9))
    Sigma  = newey_west(z, lag_nw)
    Jg     = jacobian(samp, est, h, extra_lags)
    V      = Jg @ Sigma @ Jg.T
    diag_V = np.diag(V)
    bad_keys = [['kappa','theta','sigma_V','mu','rho'][i]
                for i in range(5) if diag_V[i] <= 0]
    if bad_keys:
        raise ValueError(f"Degenerate Jacobian for {bad_keys}.")
    se     = np.sqrt(np.abs(diag_V) / N)
    z_crit = norm.ppf(1 - alpha/2)
    keys   = ['kappa', 'theta', 'sigma_V', 'mu', 'rho']
    return {key: (est[key], est[key] - z_crit*se[i],
                  est[key] + z_crit*se[i], se[i])
            for i, key in enumerate(keys)}

: 

In [ ]:
def replicated_ci(TRUE, V0, h, n_sub, n_rep=50, N=int(400_000)):
    keys    = ['mu', 'theta', 'kappa', 'sigma_V', 'rho']
    results = {key: [] for key in keys}
    covered = {key: 0  for key in keys}
    failed  = 0

    for r in range(n_rep):
        ret = heston_returns(V0, N, TRUE['mu'], TRUE['theta'], TRUE['kappa'],
                             TRUE['sigma_V'], TRUE['rho'], h, n_sub)
        try:
            est = estimate_heston_from_returns(ret, h)
            ci  = confidence_intervals(ret, est, h)
            for key in keys:
                results[key].append(est[key])
                e, lo, hi, s = ci[key]
                if lo <= TRUE[key] <= hi:
                    covered[key] += 1
        except Exception as ex:
            failed += 1
            print(f"  Replication {r+1:>3d} failed: {ex}")

    valid = n_rep - failed
    print(f"\n  Completed: {valid}/{n_rep}  ({failed} failed)\n")

    if valid == 0:
        print("  All replications failed — cannot compute statistics.")
        return results, failed

    print(f"{'Parameter':<12s}  {'True':>8s}  {'Mean':>10s}  "
          f"{'Std':>10s}  {'Bias':>10s}  {'Coverage':>10s}")
    print("-" * 65)

    for key in keys:
        vals     = np.array(results[key])
        mean     = vals.mean()
        std      = vals.std()
        bias     = mean - TRUE[key]
        cov_rate = covered[key] / valid * 100
        print(f"  {key:<10s}  {TRUE[key]:8.4f}  {mean:10.4f}  "
              f"{std:10.4f}  {bias:10.4f}  {cov_rate:>9.1f}%")

    return results, failed

: 

In [ ]:
# ── Run ───────────────────────────────────────────────────────────────────────
TRUE  = dict(mu=0.125, kappa=0.1, theta=0.25, sigma_V=0.1, rho=-0.7)
V0    = TRUE['theta']
h     = 1.0
n_sub = int(20)


results, failures = replicated_ci(TRUE, V0, h, n_sub, n_rep=100, N=int(400_000))
print(f"\nTotal failures: {failures} out of 100")

: 

In [ ]:
def experiment_v0_burnin(TRUE, h, n_sub, n_rep=100, N=int(400_000)):
    burnin  = int(5_000)
    configs = {
        'A: V0=θ (stationary)'   : (TRUE['theta'],       0),
        'B: V0=4θ (no burnin)'   : (4*TRUE['theta'],     0),
        'C: V0=0.1θ (no burnin)' : (0.1*TRUE['theta'],   0),
        'D: V0=4θ (burnin)'      : (4*TRUE['theta'],     burnin),
        'E: V0=0.1θ (burnin)'    : (0.1*TRUE['theta'],   burnin),  # NEW
    }
    results = {name: {key: [] for key in ['kappa','theta','sigma_V','mu','rho']}
               for name in configs}
    failed  = {name: 0 for name in configs}

    for name, (V0, burn) in configs.items():
        print(f"\nRunning config: {name}")
        for r in range(n_rep):
            ret_full = heston_returns(V0, N + burn,
                                      TRUE['mu'], TRUE['theta'], TRUE['kappa'],
                                      TRUE['sigma_V'], TRUE['rho'], h, n_sub)
            ret = ret_full[burn:]
            try:
                est = estimate_heston_from_returns(ret, h)
                if not np.isfinite(est['kappa']) or est['kappa'] <= 0:
                    raise ValueError(f"Invalid kappa = {est['kappa']:.4e}")
                for key in results[name]:
                    results[name][key].append(est[key])
            except Exception as ex:
                failed[name] += 1

        print(f"  Failed: {failed[name]}/{n_rep}")

    # Print bias comparison
    print(f"\n{'Config':<25s}  {'κ bias':>10s}  {'θ bias':>10s}  "
          f"{'σ_V bias':>10s}  {'μ bias':>10s}  {'ρ bias':>10s}  "
          f"{'Failed':>8s}")
    print("-" * 85)
    for name in configs:
        vals = results[name]
        biases = {key: np.nanmean(vals[key]) - TRUE[key]
                  for key in ['kappa','theta','sigma_V','mu','rho']}
        print(f"  {name:<23s}  "
              f"{biases['kappa']:10.4f}  {biases['theta']:10.4f}  "
              f"{biases['sigma_V']:10.4f}  {biases['mu']:10.4f}  "
              f"{biases['rho']:10.4f}  {failed[name]:>8d}")

    return results, failed

: 

In [ ]:
# ── Run ───────────────────────────────────────────────────────────────────────
TRUE  = dict(mu=0.125, kappa=0.1, theta=0.25, sigma_V=0.1, rho=-0.7)
V0    = TRUE['theta']
h     = 1.0
n_sub = int(20)


burnin = experiment_v0_burnin(TRUE, h, n_sub, n_rep = 100, N=int(100_000))


Running config: A: V0=θ (stationary)
  κ̂    = 0.1572
  θ̂    = 0.2453
  μ̂    = 0.1226
  σ̂_V  = 0.134885
  ρ̂    = -0.721938
  κ̂    = 0.0826
  θ̂    = 0.2515
  μ̂    = 0.1258
  σ̂_V  = 0.092996
  ρ̂    = -0.641418
  κ̂    = 0.1196
  θ̂    = 0.2494
  μ̂    = 0.1258
  σ̂_V  = 0.116946
  ρ̂    = -0.699000
  κ̂    = 0.0954
  θ̂    = 0.2503
  μ̂    = 0.1248
  σ̂_V  = 0.106427
  ρ̂    = -0.559773
  κ̂    = 0.0827
  θ̂    = 0.2509
  μ̂    = 0.1258
  σ̂_V  = 0.091935
  ρ̂    = -0.673434
  κ̂    = 0.1018
  θ̂    = 0.2502
  μ̂    = 0.1249
  σ̂_V  = 0.087400
  ρ̂    = -0.847017
  κ̂    = 0.0940
  θ̂    = 0.2519
  μ̂    = 0.1232
  σ̂_V  = 0.099062
  ρ̂    = -0.731385
  κ̂    = 0.1102
  θ̂    = 0.2492
  μ̂    = 0.1264
  σ̂_V  = 0.108287
  ρ̂    = -0.793562
  κ̂    = 0.1637
  θ̂    = 0.2513
  μ̂    = 0.1224
  σ̂_V  = 0.133961
  ρ̂    = -0.669656
  κ̂    = 0.0418
  θ̂    = 0.2492
  μ̂    = 0.1260
  σ̂_V  = 0.057321
  ρ̂    = -0.975448
  κ̂    = 0.0638
  θ̂    = 0.2547
  μ̂    = 0.1245
  σ̂_V  = 0

: 

In [ ]:
# ── Run ───────────────────────────────────────────────────────────────────────
TRUE  = dict(mu=0.125, kappa=0.01, theta=0.25, sigma_V=0.1, rho=-0.7)
V0    = TRUE['theta']
h     = 1.0
n_sub = int(20)


burnin2 = experiment_v0_burnin(TRUE, h, n_sub, n_rep = 100, N=int(100_000))


Running config: A: V0=θ (stationary)
  κ̂    = 0.0137
  θ̂    = 0.2559
  μ̂    = 0.1255
  σ̂_V  = 0.115596
  ρ̂    = -0.621096
  κ̂    = 0.0076
  θ̂    = 0.2646
  μ̂    = 0.1269
  σ̂_V  = 0.085186
  ρ̂    = -0.906167
  κ̂    = 0.0018
  θ̂    = 0.2613
  μ̂    = 0.1288
  σ̂_V  = 0.043985
  ρ̂    = -0.922758
  κ̂    = 0.0218
  θ̂    = 0.2367
  μ̂    = 0.1249
  σ̂_V  = 0.142020
  ρ̂    = -0.501046
  κ̂    = 0.0057
  θ̂    = 0.2347
  μ̂    = 0.1264
  σ̂_V  = 0.071641
  ρ̂    = -0.946196
  κ̂    = 0.0079
  θ̂    = 0.2910
  μ̂    = 0.1227
  σ̂_V  = 0.091466
  ρ̂    = -0.915565
  κ̂    = 0.0176
  θ̂    = 0.2401
  μ̂    = 0.1238
  σ̂_V  = 0.129915
  ρ̂    = -0.688942
  κ̂    = 0.0231
  θ̂    = 0.2569
  μ̂    = 0.1235
  σ̂_V  = 0.135042
  ρ̂    = -0.680608
  κ̂    = 0.0130
  θ̂    = 0.2729
  μ̂    = 0.1229
  σ̂_V  = 0.132751
  ρ̂    = -0.390366
  κ̂    = 0.0154
  θ̂    = 0.2609
  μ̂    = 0.1255
  σ̂_V  = 0.126028
  ρ̂    = -0.475535
  κ̂    = 0.0166
  θ̂    = 0.2500
  μ̂    = 0.1221
  σ̂_V  = 0

C:\Users\karis\AppData\Local\Temp\ipykernel_18032\2843906162.py:37: RuntimeWarning: invalid value encountered in scalar divide
  return (1 - np.exp(-kappa * h)) / kappa


  κ̂    = 0.0009
  θ̂    = 0.2593
  μ̂    = 0.1288
  σ̂_V  = 0.029043
  ρ̂    = -0.999000
  κ̂    = 0.0240
  θ̂    = 0.2574
  μ̂    = 0.1247
  σ̂_V  = 0.164449
  ρ̂    = -0.411580
  κ̂    = 0.0072
  θ̂    = 0.2778
  μ̂    = 0.1231
  σ̂_V  = 0.086985
  ρ̂    = -0.853478
  κ̂    = 0.0104
  θ̂    = 0.2435
  μ̂    = 0.1254
  σ̂_V  = 0.091048
  ρ̂    = -0.799175
  κ̂    = 0.0064
  θ̂    = 0.2310
  μ̂    = 0.1276
  σ̂_V  = 0.073765
  ρ̂    = -0.707747
  κ̂    = 0.0130
  θ̂    = 0.2380
  μ̂    = 0.1261
  σ̂_V  = 0.112269
  ρ̂    = -0.653717
  κ̂    = 0.0016
  θ̂    = 0.2454
  μ̂    = 0.1270
  σ̂_V  = 0.038270
  ρ̂    = -0.999000
  κ̂    = 0.0034
  θ̂    = 0.2425
  μ̂    = 0.1263
  σ̂_V  = 0.054979
  ρ̂    = -0.999000
  κ̂    = 0.0168
  θ̂    = 0.2425
  μ̂    = 0.1237
  σ̂_V  = 0.137220
  ρ̂    = -0.513729
  κ̂    = 0.0080
  θ̂    = 0.2520
  μ̂    = 0.1251
  σ̂_V  = 0.078199
  ρ̂    = -0.999000
  κ̂    = 0.0054
  θ̂    = 0.2696
  μ̂    = 0.1297
  σ̂_V  = 0.075777
  ρ̂    = -0.630749
  κ̂    = 

: 

In [ ]:

def experiment_parameter_settings(h, n_sub, n_rep=400, N=int(400_000)):
    """
    Six parameter settings,
    each differing from S0 in exactly one parameter.
    """
    settings = {
        'S0 (base)' : dict(mu=0.125, kappa=0.1,  theta=0.25, sigma_V=0.1,  rho=-0.7),
        'S1 (μ=0.4)': dict(mu=0.4,   kappa=0.1,  theta=0.25, sigma_V=0.1,  rho=-0.7),
        'S2 (κ=0.03)': dict(mu=0.125, kappa=0.03, theta=0.25, sigma_V=0.1,  rho=-0.7),
        'S3 (θ=0.5)': dict(mu=0.125, kappa=0.1,  theta=0.5,  sigma_V=0.1,  rho=-0.7),
        'S4 (σ=0.2)': dict(mu=0.125, kappa=0.1,  theta=0.25, sigma_V=0.2,  rho=-0.7),
        'S5 (ρ=-0.3)': dict(mu=0.125, kappa=0.1,  theta=0.25, sigma_V=0.1,  rho=-0.3),
    }

    for sname, TRUE in settings.items():
        print(f"\n{'='*50}")
        print(f"Setting: {sname}")
        print(f"{'='*50}")
        V0 = TRUE['theta']
        replicated_ci(TRUE, V0, h, n_sub, n_rep=n_rep, N=N)

: 

In [ ]:
# ── Run ───────────────────────────────────────────────────────────────────────
TRUE  = dict(mu=0.125, kappa=0.01, theta=0.25, sigma_V=0.1, rho=-0.7)
V0    = TRUE['theta']
h     = 1.0
n_sub = int(20)


coverage = experiment_parameter_settings(h, n_sub, n_rep = 100, N=int(100_000))

: 

In [ ]:
def experiment_feller(TRUE_base, V0, h, n_sub, n_rep=100, N=int(400_000)):
    """
    Tests estimator performance as Feller condition is approached.
    Reports ALL parameters, failure rate, and variance process behaviour.
    """
    sigma_V_vals = [0.05, 0.10, 0.20, 0.30, 0.35, 0.40, 0.50]
    keys         = ['mu', 'theta', 'kappa', 'sigma_V', 'rho']

    results = []

    for sigma_V in sigma_V_vals:
        TRUE         = {**TRUE_base, 'sigma_V': sigma_V}
        feller_ratio = 2*TRUE['kappa']*TRUE['theta'] / sigma_V**2
        satisfied    = feller_ratio >= 1

        estimates    = {k: [] for k in keys}
        failed       = 0
        v_zero_count = 0   # how often V_t hits zero during simulation

        for r in range(n_rep):
            ret = heston_returns(V0, N,
                                 TRUE['mu'], TRUE['theta'], TRUE['kappa'],
                                 sigma_V, TRUE['rho'], h, n_sub)

            # Check if variance process hit zero during simulation
            # (proxy: check if any returns are unusually small)
            if np.any(np.abs(ret) < 1e-10):
                v_zero_count += 1

            try:
                est = estimate_heston_from_returns(ret, h)
                for k in keys:
                    estimates[k].append(est[k])
            except Exception:
                failed += 1

        row = {
            'sigma_V'     : sigma_V,
            'feller_ratio': feller_ratio,
            'satisfied'   : satisfied,
            'failed'      : failed,
            'v_zero_pct'  : v_zero_count / n_rep * 100,
        }
        for k in keys:
            vals         = np.array(estimates[k])
            row[f'bias_{k}'] = vals.mean() - TRUE[k] if len(vals) > 0 else np.nan
            row[f'std_{k}']  = vals.std()             if len(vals) > 0 else np.nan
            row[f'rmse_{k}'] = np.sqrt((vals.mean()-TRUE[k])**2 + vals.var()) \
                                if len(vals) > 0 else np.nan
        results.append(row)

    # ── Print bias table for all parameters ───────────────────────────────
    print(f"\n{'σ_V':<6s}  {'Feller':>8s}  {'OK':>4s}  " +
          "  ".join(f"{'bias('+k+')':>10s}" for k in keys) +
          f"  {'Failed':>7s}  {'V→0%':>6s}")
    print("-" * 95)
    for r in results:
        print(f"  {r['sigma_V']:<4.3f}  {r['feller_ratio']:>8.3f}  "
              f"{'✓' if r['satisfied'] else '✗':>4s}  " +
              "  ".join(f"{r['bias_'+k]:>10.4f}" for k in keys) +
              f"  {r['failed']:>7d}  {r['v_zero_pct']:>5.1f}%")

    # ── Print RMSE table ───────────────────────────────────────────────────
    print(f"\n{'σ_V':<6s}  {'Feller':>8s}  " +
          "  ".join(f"{'RMSE('+k+')':>10s}" for k in keys))
    print("-" * 75)
    for r in results:
        print(f"  {r['sigma_V']:<4.3f}  {r['feller_ratio']:>8.3f}  " +
              "  ".join(f"{r['rmse_'+k]:>10.4f}" for k in keys))

    # ── Plot ───────────────────────────────────────────────────────────────
    fig, axes = plt.subplots(2, 3, figsize=(15, 9))
    fig.suptitle('Feller Condition Experiment', fontsize=12, fontweight='bold')
    axes = axes.flatten()

    sigma_V_plot  = [r['sigma_V']      for r in results]
    feller_plot   = [r['feller_ratio'] for r in results]

    for i, k in enumerate(keys):
        ax   = axes[i]
        bias = [r[f'bias_{k}'] for r in results]
        rmse = [r[f'rmse_{k}'] for r in results]

        ax.plot(sigma_V_plot, bias, 'o-', color='#1565C0',
                lw=2, label='bias', zorder=4)
        ax.plot(sigma_V_plot, rmse, 's--', color='#C62828',
                lw=2, label='RMSE', zorder=4)
        ax.axhline(0, color='black', lw=0.8, ls=':')

        # Shade Feller-violated region
        feller_threshold = np.sqrt(2*TRUE_base['kappa']*TRUE_base['theta'])
        ax.axvspan(feller_threshold, max(sigma_V_plot),
                   color='red', alpha=0.08, label='Feller violated')
        ax.axvline(feller_threshold, color='red', lw=1.5, ls='--', alpha=0.5)

        ax.set_title(k)
        ax.set_xlabel(r'$\sigma_V$')
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.3, ls=':')

    # Panel 6 — failure rate and V→0
    ax = axes[5]
    failed_pct = [r['failed']/n_rep*100   for r in results]
    vzero_pct  = [r['v_zero_pct']         for r in results]
    ax.plot(sigma_V_plot, failed_pct, 'o-', color='#C62828',
            lw=2, label='Failed (%)')
    ax.plot(sigma_V_plot, vzero_pct,  's--', color='#E65100',
            lw=2, label='V→0 (%)')
    ax.axvline(feller_threshold, color='red', lw=1.5, ls='--', alpha=0.5)
    ax.axvspan(feller_threshold, max(sigma_V_plot),
               color='red', alpha=0.08)
    ax.set_title('Failure rate & V→0')
    ax.set_xlabel(r'$\sigma_V$')
    ax.set_ylabel('%')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3, ls=':')

    plt.tight_layout()
    plt.savefig('feller_experiment.png', dpi=150, bbox_inches='tight')
    plt.show()

    return results

: 

In [ ]:
TRUE_base = dict(mu=0.125, kappa=0.1, theta=0.25, sigma_V=0.1, rho=-0.7)
V0 = TRUE_base['theta']    # start in stationarity
h = 1.0              # annual observation interval
N = int(400_000)          # number of observations
n_sub = int(20)              # sub-steps per interval — critical for stability

results = experiment_feller(TRUE_base, V0, h, n_sub, n_rep=100, N=int(400_000))

: 

In [ ]:
def experiment_rho_sweep(TRUE_base, V0, h, n_sub, n_rep=100, N=int(400_000)):
    """
    Sweeps rho from -0.9 to 0.9.
    Shows how leverage effect affects estimation of all parameters.
    Key finding: more negative rho → better identification of sigma_V
    through cross-covariance moment.
    """
    rho_vals = [-0.9, -0.7, -0.5, -0.3, 0.0, 0.3, 0.5, 0.7, 0.9]

    results = []
    for rho in rho_vals:
        TRUE  = {**TRUE_base, 'rho': rho}
        stds  = {k: [] for k in ['kappa', 'sigma_V', 'rho']}
        failed = 0

        for r in range(n_rep):
            ret = heston_returns(V0, N,
                                 TRUE['mu'], TRUE['theta'], TRUE['kappa'],
                                 TRUE['sigma_V'], rho, h, n_sub)
            try:
                est = estimate_heston_from_returns(ret, h)
                for k in ['kappa', 'sigma_V', 'rho']:
                    val = est[k]
                    if np.isfinite(val):
                        stds[k].append(est[k])
            except Exception:
                failed += 1

        results.append({
            'rho'       : rho,
            'std_kappa' : np.std(stds['kappa']) if len(stds['kappa']) > 1 else np.nan,
            'std_sigmaV': np.std(stds['sigma_V']) if len(stds['kappa']) > 1 else np.nan,
            'std_rho'   : np.std(stds['rho']) if len(stds['kappa']) > 1 else np.nan,
            'failed'    : failed
        })

    print(f"\n{'ρ':>6s}  {'std(κ)':>10s}  {'std(σ_V)':>10s}  "
          f"{'std(ρ)':>10s}  {'Failed':>8s}")
    print("-" * 52)
    for r in results:
        print(f"  {r['rho']:>4.1f}  {r['std_kappa']:>10.4f}  "
              f"{r['std_sigmaV']:>10.4f}  "
              f"{r['std_rho']:>10.4f}  {r['failed']:>8d}")

    # Plot
    fig, axes = plt.subplots(1, 3, figsize=(14, 5))
    fig.suptitle('Estimation precision vs correlation ρ', fontweight='bold')
    rho_plot = [r['rho'] for r in results]
    for ax, key, label in zip(axes,
                               ['std_kappa', 'std_sigmaV', 'std_rho'],
                               [r'std($\hat\kappa$)',
                                r'std($\hat\sigma_V$)',
                                r'std($\hat\rho$)']):
        vals = [r[key] for r in results]
        ax.plot(rho_plot, vals, 'o-', color='#1565C0', lw=2)
        ax.set_xlabel(r'$\rho$')
        ax.set_ylabel(label)
        ax.grid(True, alpha=0.3, ls=':')
        ax.axvline(0, color='grey', lw=0.8, ls='--')
    plt.tight_layout()
    plt.show()
    return results

: 

In [ ]:
TRUE_base = dict(mu=0.125, kappa=0.1, theta=0.25, sigma_V=0.1, rho=-0.7)
V0 = TRUE_base['theta']    # start in stationarity
h = 1.0              # annual observation interval
N = int(400_000)          # number of observations
n_sub = int(20)              # sub-steps per interval — critical for stability

experiment_rho_sweep(TRUE_base, V0, h, n_sub, n_rep=50, N=int(400_000))

: 

In [ ]:
def experiment_rho_sweep_v2(TRUE_base, V0, h, n_sub, n_rep=100, N=int(400_000)):

    rho_vals  = [-0.9, -0.7, -0.5, -0.3, 0.0, 0.3, 0.5, 0.7, 0.9]
    keys      = ['kappa', 'sigma_V', 'rho']
    results   = []

    for rho in rho_vals:
        TRUE      = {**TRUE_base, 'rho': rho}
        estimates = {k: [] for k in keys}
        failed    = 0

        for r in range(n_rep):
            ret = heston_returns(V0, N,
                                 TRUE['mu'], TRUE['theta'], TRUE['kappa'],
                                 TRUE['sigma_V'], rho, h, n_sub)
            try:
                est = estimate_heston_from_returns(ret, h, verbose=False)
                for k in keys:
                    if np.isfinite(est[k]):
                        estimates[k].append(est[k])
            except Exception:
                failed += 1

        true_k = {'kappa'  : TRUE_base['kappa'],
                  'sigma_V': TRUE_base['sigma_V'],
                  'rho'    : rho}
        row = {'rho': rho, 'failed': failed,
               'n_valid': len(estimates['kappa'])}

        for k in keys:
            vals = np.array(estimates[k])
            if len(vals) > 1:
                row[f'mean_{k}'] = vals.mean()
                row[f'bias_{k}'] = vals.mean() - true_k[k]
                row[f'std_{k}']  = vals.std()
                row[f'rmse_{k}'] = np.sqrt((vals.mean()-true_k[k])**2
                                            + vals.var())
            else:
                for s in ['mean','bias','std','rmse']:
                    row[f'{s}_{k}'] = np.nan
        results.append(row)

    # ── Bias and std table ─────────────────────────────────────────────────
    print(f"\n{'ρ':>6s}  {'Valid':>6s}  " +
          "  ".join(f"{'bias('+k+')':>10s}  {'std('+k+')':>9s}"
                    for k in keys))
    print("-" * 80)
    for r in results:
        print(f"  {r['rho']:>4.1f}  {r['n_valid']:>6d}  " +
              "  ".join(
                  f"{r[f'bias_{k}']:>10.4f}  {r[f'std_{k}']:>9.4f}"
                  if np.isfinite(r[f'bias_{k}']) else
                  f"{'nan':>10s}  {'nan':>9s}"
                  for k in keys))

    # ── RMSE table ─────────────────────────────────────────────────────────
    print(f"\n{'ρ':>6s}  " +
          "  ".join(f"{'RMSE('+k+')':>10s}" for k in keys))
    print("-" * 44)
    for r in results:
        print(f"  {r['rho']:>4.1f}  " +
              "  ".join(
                  f"{r[f'rmse_{k}']:>10.4f}"
                  if np.isfinite(r[f'rmse_{k}']) else
                  f"{'nan':>10s}"
                  for k in keys))

    # ── Plot: bias and std side by side ────────────────────────────────────
    fig, axes = plt.subplots(2, 3, figsize=(15, 9))
    fig.suptitle('ρ Sweep — Bias and Std of Key Parameters',
                 fontweight='bold')
    rho_plot = [r['rho'] for r in results]
    labels   = [r'$\hat\kappa$', r'$\hat\sigma_V$', r'$\hat\rho$']

    for col, (k, label) in enumerate(zip(keys, labels)):
        bias_vals = [r[f'bias_{k}'] for r in results]
        std_vals  = [r[f'std_{k}']  for r in results]

        # Top — bias
        ax = axes[0, col]
        ax.plot(rho_plot, bias_vals, 'o-', color='#1565C0', lw=2)
        ax.axhline(0, color='black', lw=0.8, ls=':')
        ax.axvline(0, color='grey',  lw=0.8, ls='--', alpha=0.5)
        ax.set_title(f'Bias — {label}', fontsize=11)
        ax.set_xlabel(r'$\rho$')
        ax.set_ylabel(f'bias({label})')
        ax.grid(True, alpha=0.3, ls=':')

        # Bottom — std
        ax = axes[1, col]
        ax.plot(rho_plot, std_vals, 's-', color='#C62828', lw=2)
        ax.axvline(0, color='grey', lw=0.8, ls='--', alpha=0.5)
        ax.set_title(f'Std — {label}', fontsize=11)
        ax.set_xlabel(r'$\rho$')
        ax.set_ylabel(f'std({label})')
        ax.grid(True, alpha=0.3, ls=':')

    plt.tight_layout()
    plt.savefig('rho_sweep_bias_std.png', dpi=150, bbox_inches='tight')
    plt.show()

    return results

: 

In [ ]:
TRUE_base = dict(mu=0.125, kappa=0.1, theta=0.25, sigma_V=0.1, rho=-0.7)

results = experiment_rho_sweep_v2(
    TRUE_base, TRUE_base['theta'],
    h=1.0, n_sub=int(20), n_rep=50, N=int(400_000)
)

: 

In [ ]:
def experiment_sample_size(TRUE, V0, h, n_sub, n_rep=50):
    """
    Replicates Table 2 — shows how accuracy improves as N increases
    at rate 1/sqrt(N) as predicted by the CLT.
    """
    sample_sizes = [25_000, 100_000, 400_000, 1_600_000]
    keys         = ['mu', 'theta', 'kappa', 'sigma_V', 'rho']
    all_results  = []

    for N in sample_sizes:
        print(f"  Running N = {N:>10,} …")
        estimates = {key: [] for key in keys}
        failed    = 0

        for r in range(n_rep):
            ret = heston_returns(V0, int(N),
                                 TRUE['mu'], TRUE['theta'], TRUE['kappa'],
                                 TRUE['sigma_V'], TRUE['rho'], h, n_sub)
            try:
                est = estimate_heston_from_returns(ret, h)
                for key in keys:
                    estimates[key].append(est[key])
            except Exception:
                failed += 1

        # Compute mean, std, bias, RMSE for each parameter
        row = {'N': N, 'failed': failed}
        for key in keys:
            vals        = np.array(estimates[key])
            row[f'mean_{key}'] = vals.mean()
            row[f'std_{key}']  = vals.std()
            row[f'bias_{key}'] = vals.mean() - TRUE[key]
            row[f'rmse_{key}'] = np.sqrt((vals.mean()-TRUE[key])**2 + vals.var())
        all_results.append(row)

    # ── Std table ───────────────────────────
    print(f"\n{'N':<12s}  " +
          "  ".join(f"{'std('+k+')':>12s}" for k in keys) +
          f"  {'κ·√N':>8s}  {'Failed':>7s}")
    print("-" * 100)
    for row in all_results:
        N     = row['N']
        scale = row['std_kappa'] * np.sqrt(N)
        print(f"  {N:<10,}  " +
              "  ".join(f"{row['std_'+k]:12.4f}" for k in keys) +
              f"  {scale:8.3f}  {row['failed']:>7d}")

    # ── Bias table ─────────────────────────────────────────────────────────
    print(f"\n{'N':<12s}  " +
          "  ".join(f"{'bias('+k+')':>12s}" for k in keys))
    print("-" * 80)
    for row in all_results:
        N = row['N']
        print(f"  {N:<10,}  " +
              "  ".join(f"{row['bias_'+k]:12.4f}" for k in keys))


    return all_results

: 

In [ ]:
# ── Run ────────────────────────────────────────────────────────────────────────
TRUE  = dict(mu=0.125, kappa=0.1, theta=0.25, sigma_V=0.1, rho=-0.7)
V0    = TRUE['theta']
h     = 1.0
n_sub = int(20)

experiment_sample_size(TRUE, V0, h, n_sub, n_rep=100)

: 

In [ ]:

TRUE_financial = dict(
    mu      = 0.05,    # 5% annual drift
    theta   = 0.04,    # long-run variance → 20% annual vol
    kappa   = 2.0,     # mean reversion — half-life ≈ 4 months
    sigma_V = 0.3,     # vol of vol
    rho     = -0.7     # leverage effect
)

# Observation frequencies
h_daily   = 1  / 252   # ~0.00397 years per step
h_weekly  = 5  / 252   # ~0.01984 years per step  
h_monthly = 21 / 252   # ~0.08333 years per step
h_quarterly = 1/4
h_biannually = 1/2
h_annually = 1
h_2 = 2
h_4 = 4

# Expected signal at each frequency
for name, h in [('Daily', h_daily), ('Weekly', h_weekly), 
                ('Monthly', h_monthly), ('Quartely', h_quarterly),
                ('Bianually', h_biannually), ('Yearly', h_annually),
                ('Every Second Year', h_2), ('Every Fourth Year', h_4)]:
    ht    = (1 - np.exp(-TRUE_financial['kappa']*h)) / TRUE_financial['kappa']
    lag1  = (ht**2 * TRUE_financial['theta'] * TRUE_financial['sigma_V'] / 2) * \
            (TRUE_financial['sigma_V'] / (4*TRUE_financial['kappa']) - TRUE_financial['rho'])
    var   = TRUE_financial['theta'] * h
    snr   = abs(lag1) / (var / np.sqrt(400_000))
    ek    = np.exp(-TRUE_financial['kappa']*h)
    print(f"{name:<10s}  h={h:.5f}  e^(-κh)={ek:.4f}  "
          f"lag1={lag1:.4e}  SNR={snr:.2f}")

: 

In [ ]:
def experiment_frequency(TRUE, V0, h_values, h_names, n_sub_dict,
                          n_rep=50, N=int(400_000)):
    """
    Replicates Table 3 of the paper but with daily/weekly/monthly
    observation frequencies instead of h = 0.5, 1, 2, 4.

    For each frequency:
      1. Simulate N observations at that frequency
      2. Estimate parameters
      3. Report mean, std, bias across n_rep replications

    Parameters
    ----------
    TRUE      : dict of true annualised parameters
    V0        : initial variance
    h_values  : list of h values  e.g. [1/252, 5/252, 21/252]
    h_names   : list of names     e.g. ['Daily', 'Weekly', 'Monthly']
    n_sub_dict: dict mapping h to n_sub  — finer sub-steps for larger h
    n_rep     : replications per frequency
    N         : observations per replication
    """
    keys        = ['mu', 'theta', 'kappa', 'sigma_V', 'rho']
    all_results = []

    for h, name in zip(h_values, h_names):
        n_sub  = n_sub_dict.get(h, 20)
        print(f"\n{'='*55}")
        print(f"Frequency: {name}  (h = {h:.5f},  n_sub = {n_sub})")
        print(f"  Total span : {N*h:.1f} years")
        print(f"  e^(-κh)    : {np.exp(-TRUE['kappa']*h):.6f}")
        print(f"  Feller     : {'✓' if 2*TRUE['kappa']*TRUE['theta'] >= TRUE['sigma_V']**2 else '✗'}")
        print(f"{'='*55}")

        estimates = {k: [] for k in keys}
        failed    = 0

        for r in range(n_rep):
            ret = heston_returns(V0, N,
                                 TRUE['mu'], TRUE['theta'], TRUE['kappa'],
                                 TRUE['sigma_V'], TRUE['rho'], h, n_sub)
            try:
                est = estimate_heston_from_returns(ret, h)
                for k in keys:
                    estimates[k].append(est[k])
            except Exception as ex:
                failed += 1

            if (r+1) % 10 == 0:
                print(f"  {r+1}/{n_rep}  failed: {failed}")

        row = {'name': name, 'h': h, 'N': N, 'failed': failed,
               'span': N*h, 'ek': np.exp(-TRUE['kappa']*h)}
        for k in keys:
            vals       = np.array(estimates[k])
            row[f'mean_{k}']  = vals.mean()  if len(vals)>0 else np.nan
            row[f'std_{k}']   = vals.std()   if len(vals)>0 else np.nan
            row[f'bias_{k}']  = vals.mean() - TRUE[k] if len(vals)>0 else np.nan
            row[f'rmse_{k}']  = np.sqrt((vals.mean()-TRUE[k])**2 + vals.var()) \
                                 if len(vals)>0 else np.nan
        all_results.append(row)

    # ── Mean ± Std table ───────────────────────────────────────────────────
    print(f"\n\n{'Frequency':<12s}  {'h':>8s}  " +
          "  ".join(f"{'mean('+k+')':>12s}" for k in keys))
    print("-" * 90)
    for row in all_results:
        print(f"  {row['name']:<10s}  {row['h']:8.5f}  " +
              "  ".join(f"{row['mean_'+k]:12.4f}" for k in keys))

    print(f"\n{'Frequency':<12s}  {'h':>8s}  " +
          "  ".join(f"{'std('+k+')':>12s}" for k in keys) +
          f"  {'Failed':>7s}")
    print("-" * 100)
    for row in all_results:
        print(f"  {row['name']:<10s}  {row['h']:8.5f}  " +
              "  ".join(f"{row['std_'+k]:12.4f}" for k in keys) +
              f"  {row['failed']:>7d}")

    # ── Bias table ─────────────────────────────────────────────────────────
    print(f"\n{'Frequency':<12s}  {'h':>8s}  " +
          "  ".join(f"{'bias('+k+')':>12s}" for k in keys))
    print("-" * 90)
    for row in all_results:
        print(f"  {row['name']:<10s}  {row['h']:8.5f}  " +
              "  ".join(f"{row['bias_'+k]:12.4f}" for k in keys))

    # ── SNR diagnostic ─────────────────────────────────────────────────────
    print(f"\nSignal-to-noise ratio for lag-1 autocovariance:")
    print(f"{'Frequency':<12s}  {'h':>8s}  {'lag1':>12s}  "
          f"{'noise':>12s}  {'SNR':>8s}  {'e^(-κh)':>10s}")
    print("-" * 68)
    for h, name in zip(h_values, h_names):
        ht   = (1 - np.exp(-TRUE['kappa']*h)) / TRUE['kappa']
        lag1 = (ht**2*TRUE['theta']*TRUE['sigma_V']/2) * \
               (TRUE['sigma_V']/(4*TRUE['kappa']) - TRUE['rho'])
        var  = TRUE['theta'] * h
        snr  = abs(lag1) / (var / np.sqrt(N))
        ek   = np.exp(-TRUE['kappa']*h)
        print(f"  {name:<10s}  {h:8.5f}  {lag1:12.4e}  "
              f"{var/np.sqrt(N):12.4e}  {snr:8.2f}  {ek:10.6f}")

    # ── LaTeX table ────────────────────────────────────────────────────────
    print(f"\n\nLaTeX table:\n")
    print(r"\begin{table}[h]")
    print(r"\centering")
    print(r"\caption{Effect of Observation Frequency on Estimation Accuracy}")
    print(r"\label{tab:frequency}")
    print(r"\begin{threeparttable}")
    print(r"\begin{tabular}{llc" + "c"*len(keys) + "}")
    print(r"\toprule")
    print(r"Frequency & $h$ & $e^{-\kappa h}$ & " +
          " & ".join([r"$\hat\mu$", r"$\hat\theta$",
                      r"$\hat\kappa$", r"$\hat\sigma_v$",
                      r"$\hat\rho$"]) + r" \\")
    print(r"\midrule")
    for row in all_results:
        vals = " & ".join(
            f"{row['mean_'+k]:.3f}$\\pm${row['std_'+k]:.3f}"
            for k in keys
        )
        print(f"  {row['name']} & ${row['h']:.5f}$ & "
              f"${row['ek']:.4f}$ & {vals} \\\\")
    print(r"\bottomrule")
    print(r"\end{tabular}")
    print(r"\begin{tablenotes}")
    print(r"\small")
    print(r"\item \textit{Notes:} Each cell reports mean $\pm$ std across "
          + str(n_rep) + r" replications of $N=" + f"{N:,}" +
          r"$ observations. All parameters are annualised: " +
          r"$\mu=" + str(TRUE['mu']) +
          r"$, $\kappa=" + str(TRUE['kappa']) +
          r"$, $\theta=" + str(TRUE['theta']) +
          r"$, $\sigma_v=" + str(TRUE['sigma_V']) +
          r"$, $\rho=" + str(TRUE['rho']) + r"$.")
    print(r"The column $e^{-\kappa h}$ measures the lag ratio signal strength ---")
    print(r"values closer to 0 indicate stronger identification of $\kappa$.")
    print(r"\end{tablenotes}")
    print(r"\end{threeparttable}")
    print(r"\end{table}")

    return all_results


# ── Run ────────────────────────────────────────────────────────────────────────
TRUE  = dict(mu=0.05, theta=0.04, kappa=2.0, sigma_V=0.3, rho=-0.7)
V0    = TRUE['theta']
n_sub = int(20)

h_daily   = 1  / 252
h_weekly  = 5  / 252
h_monthly = 21 / 252
h_quarterly = 1/4
h_biannually = 1/2
h_annually = 1
h_2 = 2
h_4 = 4

h_values = [h_daily,   h_weekly,   h_monthly, h_quarterly, h_biannually, h_annually, h_2, h_4]
h_names  = ['Daily',   'Weekly',   'Monthly', 'Quarterly', 'Biannually', 
            'Yearly', 'Every second year', 'Every fourth year']

# Sub-steps: 
# Scale n_sub inversely with h to keep dt ≈ 1/5040 constant

dt_target = 0.05
n_sub_dict = {h: max(1, int(np.ceil(h / dt_target))) for h in h_values}

results = experiment_frequency(
    TRUE       = TRUE,
    V0         = V0,
    h_values   = h_values,
    h_names    = h_names,
    n_sub_dict = n_sub_dict,
    n_rep      = 50,
    N          = int(400_000)
)


: 

In [ ]:
def experiment_lag_M(TRUE, V0, h, n_sub, n_rep=100, N=int(400_000)):
    """
    Tests sensitivity of kappa estimate to the lag limit M.
    """
    M_vals = [2, 3, 5, 7, 10, 15, 20, 30]

    print(f"{'M':>5s}  {'mean(κ)':>10s}  {'std(κ)':>10s}  "
          f"{'bias(κ)':>10s}  {'RMSE(κ)':>10s}  {'Failed':>8s}")
    print("-" * 60)

    for M in M_vals:
        kappa_ests = []
        failed     = 0

        for r in range(n_rep):
            ret = heston_returns(V0, N,
                                 TRUE['mu'], TRUE['theta'], TRUE['kappa'],
                                 TRUE['sigma_V'], TRUE['rho'], h, n_sub)
            try:
                # FIX: call estimate_kappa directly with lag_M=M
                kappa = estimate_kappa(ret, h, lag_M=M)
                if np.isfinite(kappa) and kappa > 0:
                    kappa_ests.append(kappa)
                else:
                    failed += 1
            except Exception:
                failed += 1

        if len(kappa_ests) == 0:
            print(f"  {M:>3d}  {'all failed':>10s}  {'—':>10s}  "
                  f"{'—':>10s}  {'—':>10s}  {failed:>8d}")
            continue

        vals = np.array(kappa_ests)
        bias = vals.mean() - TRUE['kappa']
        rmse = np.sqrt(bias**2 + vals.var())

        print(f"  {M:>3d}  {vals.mean():>10.4f}  {vals.std():>10.4f}  "
              f"{bias:>10.4f}  {rmse:>10.4f}  {failed:>8d}")

: 

In [ ]:
TRUE = dict(mu=0.125, kappa=0.1, theta=0.25, sigma_V=0.1, rho=-0.7)
V0 = TRUE['theta']    # start in stationarity
h = 1.0              # annual observation interval
N = int(400_000)          # number of observations
n_sub = int(20)              # sub-steps per interval — critical for stability

experiment_lag_M(TRUE, V0, h, n_sub, n_rep=100, N=int(400_000))

: 

In [ ]:
### Black Scholes Model Comparison ###
def compare_models(ret, h, TRUE):
    
    # Empirical Moments
    samp = estimate_moments(ret)
    names = ['Mean', 'Variance', 'Lag-1 Autocovariance', 'Lag-2 Autocovariance', 'Cross-Cov']
    
    # Fit Heston
    est_heston = estimate_heston_from_returns(ret, h)
    
    # Heston Predicted Moments
    pred_heston = np.array(moments(
        est_heston['mu'], est_heston['theta'], est_heston['kappa'],
        est_heston['sigma_V'], est_heston['rho'], h
    ))
    
    # Fit Black Scholes    
    theta_bs = np.var(ret, ddof = 1) / h
    mu_bs = np.mean(ret) / h + theta_bs / 2 
    
    # BS predicted moments
    pred_bs = np.array([
        h * (mu_bs - 0.5 * theta_bs),
        theta_bs * h,
        0,
        0,
        0
    ])
    
    # ── Compare ────────────────────────────────────────────────────────────
    print(f"\n{'Moment':<20s}  {'Empirical':>12s}  {'Heston':>12s}  "
          f"{'BS':>12s}  {'Err Heston':>12s}  {'Err BS':>12s}")
    print("-" * 84)

    sse_heston = 0.0
    sse_bs     = 0.0

    for i, name in enumerate(names):
        e  = samp[i]
        ph = pred_heston[i]
        pb = pred_bs[i]
        eh = abs(e - ph)
        eb = abs(e - pb)
        sse_heston += (e - ph)**2
        sse_bs     += (e - pb)**2
        print(f"  {name:<18s}  {e:+12.4e}  {ph:+12.4e}  "
              f"{pb:+12.4e}  {eh:12.4e}  {eb:12.4e}")

    print(f"\n  SSE Heston : {sse_heston:.6e}")
    print(f"  SSE BS     : {sse_bs:.6e}")
    print(f"  Winner     : {'Heston ✓' if sse_heston < sse_bs else 'BS ✓'}")

    # ── Moment ratio test ──────────────────────────────────────────────────
    # Under BS, lag1/var should be 0. Under Heston it is nonzero.
    # Test statistic: is the empirical lag1 significantly different from 0?
    N    = len(ret)
    se_lag1 = np.var(ret, ddof=1) / np.sqrt(N)   # approx std error of lag1
    t_stat  = samp[2] / se_lag1
    print(f"\n  Lag-1 / SE = {t_stat:.3f}  "
          f"({'significant' if abs(t_stat) > 1.96 else 'not significant'} at 5%)")
    print(f"  → {'Heston volatility clustering detected' if abs(t_stat) > 1.96 else 'Cannot reject BS'}")

    return {
        'est_heston': est_heston,
        'est_bs':     {'mu': mu_bs, 'theta': theta_bs},
        'pred_heston': pred_heston,
        'pred_bs':     pred_bs,
        'samp':        samp,
        'sse_heston':  sse_heston,
        'sse_bs':      sse_bs
    }

: 

In [ ]:
def replicated_comparison(TRUE, V0, h, n_sub, n_rep=50, N=int(400_000)):
    """
    For each replication:
      1. Simulate from Heston (true model)
      2. Fit both BS and Heston
      3. Record SSE for each
      4. Record whether the lag-1 test correctly rejects BS
    """
    sse_heston_all = []
    sse_bs_all     = []
    reject_bs      = []   # did the lag-1 test correctly reject BS?

    print(f"Running {n_rep} replications …")

    for r in range(n_rep):
        ret = heston_returns(V0, N,
                             TRUE['mu'], TRUE['theta'], TRUE['kappa'],
                             TRUE['sigma_V'], TRUE['rho'], h, n_sub)
        try:
            res = compare_models(ret, h, TRUE)
            sse_heston_all.append(res['sse_heston'])
            sse_bs_all.append(res['sse_bs'])

            # Test: is lag1 significantly nonzero?
            se_lag1 = np.var(ret, ddof=1) / np.sqrt(N)
            t_stat  = res['samp'][2] / se_lag1
            reject_bs.append(abs(t_stat) > 1.96)

            if (r+1) % 10 == 0:
                print(f"  Replication {r+1}/{n_rep}")
        except Exception as ex:
            print(f"  Replication {r+1} failed: {ex}")

    sse_h = np.array(sse_heston_all)
    sse_b = np.array(sse_bs_all)

    print(f"\n{'Metric':<35s}  {'Heston':>12s}  {'BS':>12s}")
    print("-" * 62)
    print(f"  {'Mean SSE':<33s}  {sse_h.mean():12.4e}  {sse_b.mean():12.4e}")
    print(f"  {'Std SSE':<33s}  {sse_h.std():12.4e}  {sse_b.std():12.4e}")
    print(f"  {'Heston wins (lower SSE)':<33s}  "
          f"{np.mean(sse_h < sse_b)*100:.1f}%")
    print(f"  {'BS correctly rejected (lag-1 test)':<33s}  "
          f"{np.mean(reject_bs)*100:.1f}%")

    return sse_h, sse_b, reject_bs

: 

In [ ]:
compare_models(ret, h, TRUE)
replicated_comparison(TRUE, V0, h, n_sub, n_rep = 50, N = int(400_000))

: 

In [ ]:
# Under H0 (BS): Cov(y_n, y_{n+1}) = 0
# Test statistic
se_lag1 = np.var(ret, ddof=1) / np.sqrt(len(ret))
t_stat  = lagm_cov_est(ret, 1) / se_lag1

# Reject H0 at 5% if |t| > 1.96
reject = abs(t_stat) > 1.96

print(f"Lag-1 estimate : {lagm_cov_est(ret, 1):.4e}")
print(f"Standard error : {se_lag1:.4e}")
print(f"t-statistic    : {t_stat:.3f}")
print(f"Reject BS      : {reject}  (|t| {'>' if reject else '<'} 1.96)")

: 

In [ ]:
def experiment_sigma_V(TRUE_base, V0_base, h, n_sub,
                        sigma_V_vals=[0, 0.025, 0.05, 0.075],
                        n_rep=100, N=int(400_000)):

    results = []

    for sigma_V in sigma_V_vals:
        print(f"\n{'='*50}")
        print(f"sigma_V = {sigma_V}")
        print(f"{'='*50}")

        TRUE = {**TRUE_base, 'sigma_V': sigma_V}
        V0   = TRUE['theta']

        heston_wins    = 0
        lag1_rejects   = 0
        sse_heston_all = []
        sse_bs_all     = []
        heston_failed  = 0
        valid_both     = 0   # replications where BOTH models could be evaluated

        for r in range(n_rep):
            ret = heston_returns(V0, N,
                                 TRUE['mu'], TRUE['theta'], TRUE['kappa'],
                                 TRUE['sigma_V'], TRUE['rho'], h, n_sub)

            # ── Empirical moments — always computable ──────────────────────
            samp = estimate_moments(ret)

            # ── BS fit — always works ──────────────────────────────────────
            theta_bs = np.var(ret, ddof=1) / h
            mu_bs    = np.mean(ret) / h + theta_bs / 2
            pred_bs  = np.array([
                h * (mu_bs - 0.5*theta_bs),
                theta_bs * h,
                0.0, 0.0, 0.0
            ])
            sse_b = float(np.sum((samp - pred_bs)**2))

            # ── Lag-1 test — always computable ────────────────────────────
            se_lag1 = np.var(ret, ddof=1) / np.sqrt(N)
            t_stat  = lagm_cov_est(ret, 1) / se_lag1
            if abs(t_stat) > 1.96:
                lag1_rejects += 1

            # ── Heston fit — may fail when sigma_V is small ───────────────
            try:
                est_h  = estimate_heston_from_returns(ret, h)
                pred_h = np.array(moments(
                    est_h['mu'], est_h['theta'], est_h['kappa'],
                    est_h['sigma_V'], est_h['rho'], h
                ))
                sse_h = float(np.sum((samp - pred_h)**2))

                sse_heston_all.append(sse_h)
                sse_bs_all.append(sse_b)

                if sse_h < sse_b:
                    heston_wins += 1
                valid_both += 1

            except Exception as ex:
                heston_failed += 1
                # Still record BS SSE even if Heston failed
                if (r + 1) % 25 == 0:
                    print(f"  Rep {r+1}: Heston failed ({ex})")

            if (r + 1) % 25 == 0:
                print(f"  {r+1}/{n_rep}  Heston wins: {heston_wins}  "
                      f"Lag-1 rejects: {lag1_rejects}  "
                      f"Heston failed: {heston_failed}")

        results.append({
            'sigma_V'       : sigma_V,
            'n_rep'         : n_rep,
            'valid_both'    : valid_both,
            'heston_failed' : heston_failed,
            'heston_wins'   : heston_wins,
            'lag1_rejects'  : lag1_rejects,
            'pct_heston'    : heston_wins  / valid_both * 100 if valid_both > 0 else np.nan,
            'pct_lag1'      : lag1_rejects / n_rep      * 100,
            'mean_sse_h'    : np.mean(sse_heston_all) if sse_heston_all else np.nan,
            'mean_sse_bs'   : np.mean(sse_bs_all)     if sse_bs_all     else np.nan,
        })

    # ── Print table ────────────────────────────────────────────────────────
    print(f"\n\n{'='*95}")
    print("Results Summary")
    print(f"{'='*95}")
    print(f"{'σ_V':<8s}  {'Valid':>6s}  {'Failed':>7s}  {'Heston Wins':>14s}  "
          f"{'Lag-1 Rejects':>15s}  {'Mean SSE Heston':>16s}  {'Mean SSE BS':>13s}")
    print("-" * 95)
    for r in results:
        ratio = (r['mean_sse_bs']/r['mean_sse_h']
                 if (r['mean_sse_h'] and r['mean_sse_h'] > 0) else np.nan)
        print(f"  {r['sigma_V']:<6.3f}  "
              f"{r['valid_both']:>6d}  "
              f"{r['heston_failed']:>7d}  "
              f"{r['heston_wins']:>5d}/{r['valid_both']:<4d} ({r['pct_heston']:5.1f}%)  "
              f"{r['lag1_rejects']:>6d}/{r['n_rep']:<4d} ({r['pct_lag1']:5.1f}%)  "
              f"{r['mean_sse_h']:>16.4e}  "
              f"{r['mean_sse_bs']:>13.4e}")
    print(f"{'='*95}")

    return results


: 

In [ ]:
def experiment_sigma_V(TRUE_base, V0_base, h, n_sub,
                        sigma_V_vals=[0, 0.025, 0.05, 0.075],
                        n_rep=100, N=int(400_000)):

    results = []

    for sigma_V in sigma_V_vals:
        print(f"\n{'='*50}")
        print(f"sigma_V = {sigma_V}")
        print(f"{'='*50}")

        TRUE = {**TRUE_base, 'sigma_V': sigma_V}
        V0   = TRUE['theta']

        heston_wins    = 0
        lag1_rejects   = 0
        sse_heston_all = []
        sse_bs_all     = []
        heston_failed  = 0
        valid_both     = 0   # replications where BOTH models could be evaluated

        for r in range(n_rep):
            ret = heston_returns(V0, N,
                                 TRUE['mu'], TRUE['theta'], TRUE['kappa'],
                                 TRUE['sigma_V'], TRUE['rho'], h, n_sub)

            # ── Empirical moments — always computable ──────────────────────
            samp = estimate_moments(ret)

            # ── BS fit — always works ──────────────────────────────────────
            theta_bs = np.var(ret, ddof=1) / h
            mu_bs    = np.mean(ret) / h + theta_bs / 2
            pred_bs  = np.array([
                h * (mu_bs - 0.5*theta_bs),
                theta_bs * h,
                0.0, 0.0, 0.0
            ])
            sse_b = float(np.sum((samp - pred_bs)**2))

            # ── Lag-1 test — always computable ────────────────────────────
            se_lag1 = np.var(ret, ddof=1) / np.sqrt(N)
            t_stat  = lagm_cov_est(ret, 1) / se_lag1
            if abs(t_stat) > 1.96:
                lag1_rejects += 1

            # ── Heston fit — may fail when sigma_V is small ───────────────
            try:
                est_h  = estimate_heston_from_returns(ret, h)
                pred_h = np.array(moments(
                    est_h['mu'], est_h['theta'], est_h['kappa'],
                    est_h['sigma_V'], est_h['rho'], h
                ))
                sse_h = float(np.sum((samp - pred_h)**2))

                sse_heston_all.append(sse_h)
                sse_bs_all.append(sse_b)

                if sse_h < sse_b:
                    heston_wins += 1
                valid_both += 1

            except Exception as ex:
                heston_failed += 1
                # Still record BS SSE even if Heston failed
                if (r + 1) % 25 == 0:
                    print(f"  Rep {r+1}: Heston failed ({ex})")

            if (r + 1) % 25 == 0:
                print(f"  {r+1}/{n_rep}  Heston wins: {heston_wins}  "
                      f"Lag-1 rejects: {lag1_rejects}  "
                      f"Heston failed: {heston_failed}")

        results.append({
            'sigma_V'       : sigma_V,
            'n_rep'         : n_rep,
            'valid_both'    : valid_both,
            'heston_failed' : heston_failed,
            'heston_wins'   : heston_wins,
            'lag1_rejects'  : lag1_rejects,
            'pct_heston'    : heston_wins  / valid_both * 100 if valid_both > 0 else np.nan,
            'pct_lag1'      : lag1_rejects / n_rep      * 100,
            'mean_sse_h'    : np.mean(sse_heston_all) if sse_heston_all else np.nan,
            'mean_sse_bs'   : np.mean(sse_bs_all)     if sse_bs_all     else np.nan,
        })

    # ── Print table ────────────────────────────────────────────────────────
    print(f"\n\n{'='*95}")
    print("Results Summary")
    print(f"{'='*95}")
    print(f"{'σ_V':<8s}  {'Valid':>6s}  {'Failed':>7s}  {'Heston Wins':>14s}  "
          f"{'Lag-1 Rejects':>15s}  {'Mean SSE Heston':>16s}  {'Mean SSE BS':>13s}")
    print("-" * 95)
    for r in results:
        ratio = (r['mean_sse_bs']/r['mean_sse_h']
                 if (r['mean_sse_h'] and r['mean_sse_h'] > 0) else np.nan)
        print(f"  {r['sigma_V']:<6.3f}  "
              f"{r['valid_both']:>6d}  "
              f"{r['heston_failed']:>7d}  "
              f"{r['heston_wins']:>5d}/{r['valid_both']:<4d} ({r['pct_heston']:5.1f}%)  "
              f"{r['lag1_rejects']:>6d}/{r['n_rep']:<4d} ({r['pct_lag1']:5.1f}%)  "
              f"{r['mean_sse_h']:>16.4e}  "
              f"{r['mean_sse_bs']:>13.4e}")
    print(f"{'='*95}")

    return results


: 

In [ ]:
# Same runs but different values of sigma_V

# ── Run ───────────────────────────────────────────────────────────────────────
TRUE_base = dict(mu=0.125, kappa=0.1, theta=0.25, sigma_V=0.1, rho=-0.7)
h         = 1.0
n_sub     = int(20)

results = experiment_sigma_V(
    TRUE_base   = TRUE_base,
    V0_base     = TRUE_base['theta'],
    h           = h,
    n_sub       = n_sub,
    sigma_V_vals = [0, 0.00625, 0.0125, 0.025, 0.05],
    n_rep        = 100,
    N            = int(400_000)
)

: 

In [ ]:
def goodness_of_fit_test(ret_real, h, n_boot=200):

    # Estimate and validate
    est  = estimate_heston_from_returns(ret_real, h, verbose=False)
    samp = estimate_moments(ret_real)

    # Guard: check all parameters are finite
    for k, v in est.items():
        if not np.isfinite(v):
            raise ValueError(
                f"Parameter {k} = {v} is not finite. "
                f"Estimation failed on this data — check sample size and frequency."
            )

    N     = len(ret_real)
    n_sub = int(20)
    V0    = float(est['theta'])

    print(f"Fitted parameters:")
    for k, v in est.items():
        print(f"  {k:<10s} = {v:.4f}")
    print(f"\nSimulating {n_boot} paths of N={N:,} …")

    boot_moments = []
    failed       = 0

    for b in range(n_boot):
        try:
            ret_sim = heston_returns(
                V0, int(N),
                float(est['mu']), float(est['theta']),
                float(est['kappa']), float(est['sigma_V']),
                float(est['rho']), float(h), n_sub
            )
            # Guard: check simulated returns are finite
            if not np.all(np.isfinite(ret_sim)):
                failed += 1
                continue
            m = estimate_moments(ret_sim)
            if np.all(np.isfinite(m)):
                boot_moments.append(m)
            else:
                failed += 1
        except Exception:
            failed += 1

        if (b+1) % 50 == 0:
            print(f"  {b+1}/{n_boot}  failed: {failed}")

    if len(boot_moments) == 0:
        raise ValueError("All bootstrap replications failed.")

    boot_moments = np.array(boot_moments)
    names        = ['Mean', 'Variance', 'Lag-1', 'Lag-2', 'Cross-Cov']

    print(f"\nGoodness of Fit — {len(boot_moments)} valid replications")
    print(f"{'Moment':<20s}  {'Empirical':>12s}  {'Boot Mean':>12s}  "
          f"{'Boot Std':>10s}  {'p-value':>10s}  {'Adequate?':>10s}")
    print("-" * 78)

    for i, name in enumerate(names):
        emp    = samp[i]
        b_mean = np.nanmean(boot_moments[:, i])
        b_std  = np.nanstd(boot_moments[:, i])
        if not np.isfinite(emp) or b_std == 0:
            print(f"  {name:<18s}  {'nan':>12s}  {'nan':>12s}  "
                  f"{'nan':>10s}  {'nan':>10s}  {'—':>10s}")
            continue
        p_val = 2 * min(
            np.mean(boot_moments[:, i] >= emp),
            np.mean(boot_moments[:, i] <= emp)
        )
        ok = '✓' if p_val >= 0.05 else '✗ misspecified'
        print(f"  {name:<18s}  {emp:>12.4e}  {b_mean:>12.4e}  "
              f"{b_std:>10.4e}  {p_val:>10.3f}  {ok:>10s}")

    return boot_moments

: 

In [ ]:
def goodness_of_fit_test_robust(ret_real, h, n_boot=200, 
                                 moments_to_test=None):
    """
    Robust GoF test that:
    1. Uses Bonferroni correction for multiple testing
    2. Allows restricting to well-identified moments
    3. Reports individual and joint p-values
    
    moments_to_test: list of indices to test, e.g. [0,1] for mean+var only
                     None = test all five
    """
    if moments_to_test is None:
        moments_to_test = [0, 1, 2, 3, 4]

    n_test = len(moments_to_test)
    alpha  = 0.05 / n_test   # Bonferroni correction

    est  = estimate_heston_from_returns(ret_real, h, verbose=False)
    samp = estimate_moments(ret_real)
    N    = len(ret_real)
    V0   = float(est['theta'])

    # Check all parameters are finite
    for k, v in est.items():
        if not np.isfinite(v):
            raise ValueError(f"Parameter {k} = {v} not finite.")

    boot_moments = []
    failed = 0
    for b in range(n_boot):
        try:
            ret_sim = heston_returns(V0, int(N),
                                     float(est['mu']),   float(est['theta']),
                                     float(est['kappa']),float(est['sigma_V']),
                                     float(est['rho']),  float(h), int(20))
            if np.all(np.isfinite(ret_sim)):
                m = estimate_moments(ret_sim)
                if np.all(np.isfinite(m)):
                    boot_moments.append(m)
        except Exception:
            failed += 1

    if len(boot_moments) < 10:
        raise ValueError("Too few valid bootstrap replications.")

    boot_moments = np.array(boot_moments)
    names_all    = ['Mean', 'Variance', 'Lag-1', 'Lag-2', 'Cross-Cov']

    print(f"\nGoF Test  (Bonferroni α = {alpha:.4f} per moment, "
          f"testing {n_test} moments)")
    print(f"{'Moment':<20s}  {'Empirical':>12s}  {'Boot Mean':>12s}  "
          f"{'Boot Std':>10s}  {'p-value':>10s}  {'Adequate?':>10s}")
    print("-" * 80)

    any_reject = False
    for i in moments_to_test:
        emp    = samp[i]
        b_mean = np.nanmean(boot_moments[:, i])
        b_std  = np.nanstd(boot_moments[:, i])
        if b_std == 0 or not np.isfinite(emp):
            continue
        p_val  = 2 * min(np.mean(boot_moments[:,i] >= emp),
                         np.mean(boot_moments[:,i] <= emp))
        ok     = '✓' if p_val >= alpha else '✗ misspecified'
        if p_val < alpha:
            any_reject = True
        print(f"  {names_all[i]:<18s}  {emp:>12.4e}  {b_mean:>12.4e}  "
              f"{b_std:>10.4e}  {p_val:>10.3f}  {ok:>10s}")

    print(f"\n  Joint conclusion: "
          f"{'H0 rejected — Heston misspecified' if any_reject else 'H0 not rejected — Heston adequate'}")

    return boot_moments

: 

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np

def load_real_data(ticker, start, end, freq='monthly'):
    """
    Robustly loads price data and computes returns.
    Uses proper resampling instead of naive slicing.
    """
    raw = yf.download(ticker, start=start, end=end,
                      auto_adjust=True, progress=False)

   
    if isinstance(raw.columns, pd.MultiIndex):
        raw.columns = raw.columns.droplevel(1)

    prices = raw['Close'].dropna()

    
    if freq == 'monthly':
        prices_agg = prices.resample('ME').last().dropna()
        h          = 21 / 252
    elif freq == 'weekly':
        prices_agg = prices.resample('W-FRI').last().dropna()
        h          = 5 / 252
    else:
        prices_agg = prices
        h          = 1 / 252

    ret = np.diff(np.log(prices_agg.values))

    
    ret = ret[np.isfinite(ret)]

    print(f"{ticker} ({freq}): {len(prices_agg):,} prices → "
          f"{len(ret):,} returns")
    print(f"  NaN in ret: {np.isnan(ret).sum()}")
    print(f"  Ann. return: {np.mean(ret)*(252/21)*100:.2f}%")
    print(f"  Ann. vol   : {np.std(ret)*np.sqrt(252/21)*100:.2f}%")

    return ret, h


def run_real_data_analysis(ticker, start, end):
    """
    Full pipeline: load → estimate → goodness of fit test.
    """
    print(f"\n{'='*60}")
    print(f"Ticker: {ticker}  ({start} to {end})")
    print(f"{'='*60}")

    ret, h = load_real_data(ticker, start, end, freq='monthly')

    if len(ret) < 50:
        print(f"  Too few observations ({len(ret)}) — skipping.")
        return None

    # Estimate
    print(f"\nEstimating Heston parameters …")
    try:
        est = estimate_heston_from_returns(ret, h, verbose=False)
        ann_vol  = np.sqrt(abs(est['theta'])) * 100
        halflife = np.log(2) / est['kappa'] * 252 if est['kappa'] > 0 else np.inf
        print(f"  μ       = {est['mu']:.4f}  (ann. drift = {(est['mu'] - 0.5*est['theta']) * 100:.2f}% )")
        print(f"  θ       = {est['theta']:.4f}  (implied vol ≈ {ann_vol:.1f}%)")
        print(f"  κ       = {est['kappa']:.4f}  (half-life ≈ {halflife:.0f} days)")
        print(f"  σ_V     = {est['sigma_V']:.4f}")
        print(f"  ρ       = {est['rho']:.4f}")
        feller = 2*est['kappa']*est['theta'] >= est['sigma_V']**2
        print(f"  Feller  : {'✓' if feller else '✗'}")

        # Goodness of fit
        print(f"\nRunning goodness of fit test …")
        boot = goodness_of_fit_test_robust(ret, h=h, n_boot=200)
        return {'est': est, 'boot': boot}

    except Exception as ex:
        print(f"  Estimation failed: {ex}")
        return None


# ── Run ────────────────────────────────────────────────────────────────────────
for ticker in ['SPY', 'GLD', 'TLT']:
    run_real_data_analysis(ticker, start='2005-01-01', end='2024-01-01')

: 

In [ ]:
def full_eda(tickers, start, end, freq='monthly'):
    """
    Full EDA pipeline for multiple assets.
    Covers: price paths, returns, distributional properties,
    autocorrelation, volatility clustering, outliers.
    """
    
    import yfinance as yf
    import pandas as pd
    import numpy as np
    import matplotlib.pyplot as plt
    import matplotlib.gridspec as gridspec
    from scipy import stats
    from scipy.stats import jarque_bera, shapiro, kurtosis, skew
    import statsmodels.api as sm
    import warnings
    warnings.filterwarnings('ignore')
    
    # ── Load data ──────────────────────────────────────────────────────────
    data = {}
    for ticker in tickers:
        raw = yf.download(ticker, start=start, end=end,
                          auto_adjust=True, progress=False)
        if isinstance(raw.columns, pd.MultiIndex):
            raw.columns = raw.columns.droplevel(1)
        prices = raw['Close'].dropna()
        monthly = prices.resample('ME').last().dropna()
        ret     = pd.Series(
            np.diff(np.log(monthly.values)),
            index=monthly.index[1:]
        )
        data[ticker] = {'prices': monthly, 'ret': ret}
        print(f"  {ticker}: {len(monthly)} monthly prices, "
              f"{len(ret)} returns")

    # ══════════════════════════════════════════════════════════════════════
    # TABLE 1 — Descriptive statistics
    # ══════════════════════════════════════════════════════════════════════
    print(f"\n{'='*70}")
    print("TABLE 1 — Descriptive Statistics (Monthly Log Returns)")
    print(f"{'='*70}")

    stats_rows = []
    for ticker, d in data.items():
        ret = d['ret'].values
        N   = len(ret)
        jb_stat, jb_p    = jarque_bera(ret)
        lb_res    = sm.stats.acorr_ljungbox(ret**2, lags=[12]) # sm not scipy
        lb_stat   = float(lb_res['lb_stat'].iloc[0])
        lb_p_val      = float(lb_res['lb_pvalue'].iloc[0])

        row = {
            'Ticker'       : ticker,
            'N'            : N,
            'Mean (ann.)'  : np.mean(ret)*(252/21)*100,
            'Std (ann.)'   : np.std(ret, ddof=1)*np.sqrt(252/21)*100,
            'Min'          : np.min(ret)*100,
            'Max'          : np.max(ret)*100,
            'Skewness'     : skew(ret),
            'Exc. Kurtosis': kurtosis(ret, fisher=True),
            'JB stat'      : jb_stat,
            'JB p-value'   : jb_p,
            'LB²(12) p'    : lb_p_val,   # Ljung-Box on squared returns
        }
        stats_rows.append(row)

    df_stats = pd.DataFrame(stats_rows).set_index('Ticker')

    print(f"\n{'Statistic':<20s}" +
          "".join(f"{t:>12s}" for t in tickers))
    print("-" * (20 + 12*len(tickers)))

    display_rows = [
        ('N', 'N', '{:.0f}'),
        ('Mean (ann.)', 'Ann. Return (%)', '{:.2f}'),
        ('Std (ann.)', 'Ann. Volatility (%)', '{:.2f}'),
        ('Min', 'Min Return (%)', '{:.2f}'),
        ('Max', 'Max Return (%)', '{:.2f}'),
        ('Skewness', 'Skewness', '{:.4f}'),
        ('Exc. Kurtosis', 'Excess Kurtosis', '{:.4f}'),
        ('JB stat', 'JB Statistic', '{:.2f}'),
        ('JB p-value', 'JB p-value', '{:.4f}'),
        ('LB²(12) p', 'LB²(12) p-value', '{:.4f}'),
    ]

    print(f"\n{'Statistic':<22s}" +
          "".join(f"{t:>12s}" for t in tickers))
    print("-" * (22 + 12*len(tickers)))

    for col, label, fmt in display_rows:
        row_str = f"  {label:<20s}"
        for ticker in tickers:
            val = df_stats.loc[ticker, col]
            try:
                row_str += f"{fmt.format(float(val)):>12s}"
            except (TypeError, ValueError):
                row_str += f"{str(val):>12s}"
        print(row_str) 
    # ══════════════════════════════════════════════════════════════════════
    # TABLE 2 — Autocorrelation structure
    # ══════════════════════════════════════════════════════════════════════
    print(f"\n{'='*70}")
    print("TABLE 2 — Autocorrelation Structure")
    print(f"{'='*70}")
    print(f"\n{'Lag':>5s}  " +
          "  ".join(f"{'AC('+t+')':>8s}  {'AC²('+t+')':>9s}"
                    for t in tickers))
    print("-" * (7 + 19*len(tickers)))

    for lag in [1, 2, 3, 4, 5, 6, 12]:
        row_str = f"  {lag:>3d}  "
        for ticker, d in data.items():
            ret  = d['ret'].values
            ac_r = np.corrcoef(ret[:-lag], ret[lag:])[0,1]
            ac_r2= np.corrcoef(ret[:-lag]**2, ret[lag:]**2)[0,1]
            row_str += f"{ac_r:>8.4f}  {ac_r2:>9.4f}  "
        print(row_str)

    # ══════════════════════════════════════════════════════════════════════
    # TABLE 3 — Outlier summary
    # ══════════════════════════════════════════════════════════════════════
    print(f"\n{'='*70}")
    print("TABLE 3 — Outlier Summary (|return| > 2σ and > 3σ)")
    print(f"{'='*70}")
    print(f"\n{'Ticker':<8s}  {'σ':>8s}  {'>2σ count':>10s}  "
          f"{'>2σ (%)':>8s}  {'>3σ count':>10s}  {'>3σ (%)':>8s}")
    print("-" * 56)

    for ticker, d in data.items():
        ret   = d['ret'].values
        sigma = np.std(ret, ddof=1)
        n2    = np.sum(np.abs(ret) > 2*sigma)
        n3    = np.sum(np.abs(ret) > 3*sigma)
        print(f"  {ticker:<6s}  {sigma*100:>7.2f}%  "
              f"{n2:>10d}  {n2/len(ret)*100:>7.2f}%  "
              f"{n3:>10d}  {n3/len(ret)*100:>7.2f}%")

    # ══════════════════════════════════════════════════════════════════════
    # FIGURE 1 — Price paths and returns
    # ══════════════════════════════════════════════════════════════════════
    fig, axes = plt.subplots(3, 2, figsize=(14, 12))
    fig.suptitle('Figure 1 — Price Paths and Monthly Log Returns',
                 fontsize=13, fontweight='bold')

    colors = {'SPY': '#1565C0', 'GLD': '#2E7D32', 'TLT': '#AD1457'}

    for row, (ticker, d) in enumerate(data.items()):
        prices = d['prices']
        ret    = d['ret']
        col    = colors[ticker]

        # Price path
        ax = axes[row, 0]
        ax.plot(prices.index, prices.values, lw=1.2, color=col)
        ax.set_title(f'{ticker} — Monthly Price', fontsize=10)
        ax.set_ylabel('Price (USD)')
        ax.grid(True, alpha=0.3, ls=':')

        # Returns with ±2σ bands
        ax  = axes[row, 1]
        sig = ret.std()
        ax.plot(ret.index, ret.values*100, lw=0.8,
                color=col, alpha=0.8)
        ax.axhline(0,      color='black', lw=0.8, ls='-', alpha=0.5)
        ax.axhline( 2*sig*100, color='red', lw=1.0, ls='--',
                   alpha=0.6, label='±2σ')
        ax.axhline(-2*sig*100, color='red', lw=1.0, ls='--', alpha=0.6)
        ax.axhline( 3*sig*100, color='darkred', lw=1.0, ls=':',
                   alpha=0.6, label='±3σ')
        ax.axhline(-3*sig*100, color='darkred', lw=1.0, ls=':',
                   alpha=0.6)

        # Mark outliers
        outliers = ret[np.abs(ret) > 2*sig]
        ax.scatter(outliers.index, outliers.values*100,
                   color='red', s=30, zorder=5, alpha=0.8)
        ax.set_title(f'{ticker} — Monthly Returns (%)', fontsize=10)
        ax.set_ylabel('Return (%)')
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.3, ls=':')

    plt.tight_layout()
    plt.savefig('eda_fig1_prices.png', dpi=150, bbox_inches='tight')
    plt.show()

    # ══════════════════════════════════════════════════════════════════════
    # FIGURE 2 — Distributional analysis
    # ══════════════════════════════════════════════════════════════════════
    fig, axes = plt.subplots(3, 3, figsize=(15, 12))
    fig.suptitle('Figure 2 — Distributional Analysis',
                 fontsize=13, fontweight='bold')

    for row, (ticker, d) in enumerate(data.items()):
        ret = d['ret'].values
        col = colors[ticker]

        # Histogram with normal overlay
        ax = axes[row, 0]
        ax.hist(ret*100, bins=30, density=True,
                color=col, alpha=0.6, edgecolor='white')
        x   = np.linspace(ret.min()*100, ret.max()*100, 200)
        pdf = stats.norm.pdf(x, ret.mean()*100, ret.std()*100)
        ax.plot(x, pdf, 'k-', lw=2, label='Normal fit')
        ax.set_title(f'{ticker} — Return Distribution', fontsize=10)
        ax.set_xlabel('Return (%)')
        ax.set_ylabel('Density')
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.3, ls=':')

        # QQ plot
        ax = axes[row, 1]
        (osm, osr), (slope, intercept, r) = stats.probplot(
            ret, dist='norm')
        ax.scatter(osm, osr, color=col, s=15, alpha=0.7)
        ax.plot(osm, slope*np.array(osm)+intercept,
                'k-', lw=1.5, label=f'R²={r**2:.4f}')
        ax.set_title(f'{ticker} — QQ Plot', fontsize=10)
        ax.set_xlabel('Theoretical quantiles')
        ax.set_ylabel('Sample quantiles')
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.3, ls=':')

        # Rolling volatility
        ax  = axes[row, 2]
        ret_s  = d['ret']
        roll_v = ret_s.rolling(12).std() * np.sqrt(12) * 100
        ax.plot(ret_s.index, roll_v, lw=1.2, color=col)
        ax.axhline(ret_s.std()*np.sqrt(12)*100,
                   color='black', lw=1.0, ls='--',
                   alpha=0.7, label='Full-sample vol')
        ax.set_title(f'{ticker} — Rolling 12m Volatility (%)',
                     fontsize=10)
        ax.set_ylabel('Volatility (%)')
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.3, ls=':')

    plt.tight_layout()
    plt.savefig('eda_fig2_distributions.png', dpi=150, bbox_inches='tight')
    plt.show()

    # ══════════════════════════════════════════════════════════════════════
    # FIGURE 3 — Autocorrelation and volatility clustering
    # ══════════════════════════════════════════════════════════════════════
    fig, axes = plt.subplots(3, 3, figsize=(15, 12))
    fig.suptitle('Figure 3 — Autocorrelation and Volatility Clustering',
                 fontsize=13, fontweight='bold')
    max_lag = 20

    for row, (ticker, d) in enumerate(data.items()):
        ret = d['ret'].values
        col = colors[ticker]
        N   = len(ret)
        ci  = 1.96 / np.sqrt(N)   # 95% confidence band

        # ACF of returns
        ax   = axes[row, 0]
        acf  = [np.corrcoef(ret[:-l], ret[l:])[0,1]
                for l in range(1, max_lag+1)]
        lags = range(1, max_lag+1)
        ax.bar(lags, acf, color=col, alpha=0.7, width=0.6)
        ax.axhline( ci, color='red', lw=1.2, ls='--', alpha=0.7,
                   label='95% CI')
        ax.axhline(-ci, color='red', lw=1.2, ls='--', alpha=0.7)
        ax.axhline(0,   color='black', lw=0.8)
        ax.set_title(f'{ticker} — ACF of Returns', fontsize=10)
        ax.set_xlabel('Lag (months)')
        ax.set_ylabel('Autocorrelation')
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.3, ls=':')

        # ACF of squared returns (volatility clustering)
        ax    = axes[row, 1]
        ret2  = ret**2
        acf2  = [np.corrcoef(ret2[:-l], ret2[l:])[0,1]
                 for l in range(1, max_lag+1)]
        ax.bar(lags, acf2, color=col, alpha=0.7, width=0.6)
        ax.axhline( ci, color='red', lw=1.2, ls='--', alpha=0.7,
                   label='95% CI')
        ax.axhline(-ci, color='red', lw=1.2, ls='--', alpha=0.7)
        ax.axhline(0,   color='black', lw=0.8)
        ax.set_title(f'{ticker} — ACF of Squared Returns\n'
                     f'(Volatility clustering signal)', fontsize=10)
        ax.set_xlabel('Lag (months)')
        ax.set_ylabel('Autocorrelation')
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.3, ls=':')

        # Scatter: y²_t vs y_{t+1} (cross-covariance)
        ax  = axes[row, 2]
        y2  = ret[:-1]**2
        yf  = ret[1:]
        rho_cross = np.corrcoef(y2, yf)[0, 1]
        ax.scatter(y2*100, yf*100, color=col, s=8, alpha=0.4)
        ax.axhline(0, color='black', lw=0.8, ls='-', alpha=0.5)
        ax.axvline(0, color='black', lw=0.8, ls='-', alpha=0.5)
        # Regression line
        b = np.polyfit(y2, yf, 1)
        xr = np.linspace(y2.min(), y2.max(), 100)
        ax.plot(xr*100, np.polyval(b, xr)*100, 'k-', lw=1.5,
                label=f'ρ(y²,y+1)={rho_cross:.3f}')
        ax.set_title(f'{ticker} — Cross-Covariance\n'
                     f'y²_t vs y_{{t+1}} (leverage effect)',
                     fontsize=10)
        ax.set_xlabel('y²_t (%²)')
        ax.set_ylabel('y_{t+1} (%)')
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.3, ls=':')

    plt.tight_layout()
    plt.savefig('eda_fig3_autocorr.png', dpi=150, bbox_inches='tight')
    plt.show()

    # ══════════════════════════════════════════════════════════════════════
    # FIGURE 4 — Cross-asset comparison
    # ══════════════════════════════════════════════════════════════════════
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    fig.suptitle('Figure 4 — Cross-Asset Comparison',
                 fontsize=13, fontweight='bold')

    # Normalised price paths
    ax = axes[0]
    for ticker, d in data.items():
        prices_norm = d['prices'] / d['prices'].iloc[0] * 100
        ax.plot(prices_norm.index, prices_norm.values,
                lw=1.5, color=colors[ticker], label=ticker)
    ax.axhline(100, color='black', lw=0.8, ls='--', alpha=0.5)
    ax.set_title('Normalised Price Paths (base=100)')
    ax.set_ylabel('Index')
    ax.legend()
    ax.grid(True, alpha=0.3, ls=':')

    # Return correlation matrix
    ax = axes[1]
    ret_df = pd.DataFrame(
        {ticker: d['ret'] for ticker, d in data.items()}
    ).dropna()
    corr = ret_df.corr()
    im   = ax.imshow(corr.values, cmap='RdYlGn',
                     vmin=-1, vmax=1, aspect='auto')
    ax.set_xticks(range(len(tickers)))
    ax.set_yticks(range(len(tickers)))
    ax.set_xticklabels(tickers)
    ax.set_yticklabels(tickers)
    for i in range(len(tickers)):
        for j in range(len(tickers)):
            ax.text(j, i, f'{corr.values[i,j]:.3f}',
                    ha='center', va='center', fontsize=11,
                    color='black')
    plt.colorbar(im, ax=ax)
    ax.set_title('Return Correlation Matrix')

    # Volatility comparison — rolling 12m
    ax = axes[2]
    for ticker, d in data.items():
        roll_v = d['ret'].rolling(12).std() * np.sqrt(12) * 100
        ax.plot(roll_v.index, roll_v.values,
                lw=1.2, color=colors[ticker],
                label=ticker, alpha=0.8)
    ax.set_title('Rolling 12m Volatility (%)')
    ax.set_ylabel('Volatility (%)')
    ax.legend()
    ax.grid(True, alpha=0.3, ls=':')

    plt.tight_layout()
    plt.savefig('eda_fig4_comparison.png', dpi=150, bbox_inches='tight')
    plt.show()

    # ══════════════════════════════════════════════════════════════════════
    # TABLE 4 — Moment estimates (connects EDA to Heston estimation)
    # ══════════════════════════════════════════════════════════════════════
    print(f"\n{'='*70}")
    print("TABLE 4 — Empirical Moments (inputs to MM estimator)")
    print(f"{'='*70}")
    print(f"\n{'Moment':<25s}" +
          "".join(f"{t:>14s}" for t in tickers))
    print("-" * (25 + 14*len(tickers)))

    moment_labels = [
        'Mean return (×10⁻³)',
        'Variance (×10⁻³)',
        'Lag-1 autocov (×10⁻⁵)',
        'Lag-2 autocov (×10⁻⁵)',
        'Cross-cov (×10⁻⁵)',
    ]
    scales = [1e3, 1e3, 1e5, 1e5, 1e5]

    for i, (label, scale) in enumerate(zip(moment_labels, scales)):
        row_str = f"  {label:<23s}"
        for ticker, d in data.items():
            ret = d['ret'].values
            m   = estimate_moments(ret)
            row_str += f"{m[i]*scale:>14.4f}"
        print(row_str)

    # Lag-1 test for each asset
    print(f"\n{'='*70}")
    print("TABLE 5 — Lag-1 Test for Stochastic Volatility (H₀: BS)")
    print(f"{'='*70}")
    print(f"\n{'Statistic':<20s}" +
          "".join(f"{t:>12s}" for t in tickers))
    print("-" * (20 + 12*len(tickers)))

    for label, fn in [
        ('Lag-1 autocov', lambda r: lagm_cov_est(r, 1)),
        ('SE',            lambda r: np.var(r,ddof=1)/np.sqrt(len(r))),
        ('t-statistic',   lambda r: lagm_cov_est(r,1) /
                                    (np.var(r,ddof=1)/np.sqrt(len(r)))),
        ('p-value',       lambda r: 2*(1-stats.norm.cdf(
                           abs(lagm_cov_est(r,1) /
                               (np.var(r,ddof=1)/np.sqrt(len(r))))))),
        ('Reject BS?',    None),
    ]:
        row_str = f"  {label:<18s}"
        for ticker, d in data.items():
            ret = d['ret'].values
            if label == 'Reject BS?':
                t   = lagm_cov_est(ret,1) / (np.var(ret,ddof=1)/np.sqrt(len(ret)))
                p   = 2*(1-stats.norm.cdf(abs(t)))
                row_str += f"{'Yes' if p < 0.05 else 'No':>12s}"
            else:
                val = fn(ret)
                row_str += f"{val:>12.4e}"
        print(row_str)

    return data


# ── Run ────────────────────────────────────────────────────────────────────────
data = full_eda(
    tickers = ['SPY', 'GLD', 'TLT'],
    start   = '2005-01-01',
    end     = '2024-01-01',
    freq    = 'monthly'
)

: 

In [ ]:
def diagnose_real_data(ret, h, ticker):
    """
    Prints diagnostics before estimation to understand
    why the estimator might struggle on real data.
    """
    samp = estimate_moments(ret)
    mean_e, var_e, lag1_e, lag2_e, cross_e = samp
    N = len(ret)

    print(f"\nDiagnostics for {ticker}:")
    print(f"  N observations   : {N}")
    print(f"  h                : {h:.5f}")
    print(f"  Mean return      : {mean_e:+.6e}")
    print(f"  Variance         : {var_e:+.6e}")
    print(f"  Lag-1 autocov    : {lag1_e:+.6e}")
    print(f"  Lag-2 autocov    : {lag2_e:+.6e}")
    print(f"  Cross-cov        : {cross_e:+.6e}")

    # Lag ratio
    if lag1_e != 0 and np.sign(lag1_e) == np.sign(lag2_e):
        ratio = lag2_e / lag1_e
        if 0 < ratio < 1:
            kappa_est = -np.log(ratio) / h
            print(f"  Lag ratio        : {ratio:.4f}  → κ ≈ {kappa_est:.4f}")
        else:
            print(f"  Lag ratio        : {ratio:.4f}  ✗ not in (0,1)")
    else:
        print(f"  Lag ratio        : invalid (sign mismatch or zero)")

    # Noise floor
    noise = var_e / np.sqrt(N)
    snr   = abs(lag1_e) / noise if noise > 0 else 0
    print(f"  Noise floor      : {noise:.4e}")
    print(f"  SNR (lag1/noise) : {snr:.2f}  "
          f"({'adequate' if snr > 2 else 'too low — estimates unreliable'})")

    # Check cross-cov sign — determines sigma_sq sign
    print(f"  Cross-cov sign   : {'positive' if cross_e > 0 else 'negative'}")
    print(f"  Lag1 sign        : {'positive' if lag1_e > 0 else 'negative'}")

: 

In [ ]:
# Run diagnostics first
ret_spy, h_m = load_real_data('SPY', '2005-01-01', '2024-01-01', 'monthly')
diagnose_real_data(ret_spy, h_m, 'SPY')

ret_gld, h_m = load_real_data('GLD', '2005-01-01', '2024-01-01', 'monthly')
diagnose_real_data(ret_gld, h_m, 'GLD')

ret_tlt, h_m = load_real_data('TLT', '2005-01-01', '2024-01-01', 'monthly')
diagnose_real_data(ret_tlt, h_m, 'TLT')

: 

In [ ]:
def lag_autocov_test(ret, h, test_lags=[3, 4, 5], alpha=0.05):
    """
    Tests whether higher-order autocovariances are zero.
    
    H0: Cov(y_n, y_{n+m}) = 0  for m in test_lags  (consistent with BS)
    H1: Cov(y_n, y_{n+m}) != 0                       (consistent with Heston)
    
    No parameter estimation needed — correct size by construction.
    Under H0: Z_m = lag_m / (var_y/sqrt(N)) ~ N(0,1)
    """
    from scipy.stats import norm

    N     = len(ret)
    var_y = np.var(ret, ddof=1)
    se    = var_y / np.sqrt(N)   # Bartlett SE for zero-autocorrelation null

    print(f"\nAutocovariance Test  (N={N:,})")
    print(f"H0: Cov(y_n, y_{{n+m}}) = 0  for m = {test_lags}")
    print(f"\n{'Lag':>5s}  {'lag_m':>14s}  {'SE':>12s}  "
          f"{'Z-stat':>8s}  {'p-value':>10s}  {'OK?':>5s}")
    print("-" * 62)

    p_vals = []
    for m in test_lags:
        lagm  = lagm_cov_est(ret, m)
        z_m   = lagm / se
        p_val = 2*(1 - norm.cdf(abs(z_m)))
        p_vals.append(p_val)
        ok    = '✓' if p_val >= alpha/len(test_lags) else '✗'
        print(f"  {m:>3d}  {lagm:>14.6e}  {se:>12.4e}  "
              f"{z_m:>8.3f}  {p_val:>10.4f}  {ok:>5s}")

    valid_p = [p for p in p_vals if np.isfinite(p)]
    bon_p   = min(min(valid_p)*len(valid_p), 1.0) if valid_p else np.nan
    verdict = '✓ Cannot reject H0' if bon_p >= alpha else '✗ Reject H0'
    print(f"\n  Bonferroni p-value: {bon_p:.4f}  {verdict}")
    return p_vals


def power_analysis_lag_test(TRUE_heston, h, n_sim=500, N_vals=None):
    """
    Power analysis for the autocovariance test.
    Correct size because no parameters are estimated.
    """
    from scipy.stats import norm

    if N_vals is None:
        N_vals = [50, 100, 200, 500, 1000, 2000, 5000, 10000, 50000]

    n_sub     = int(20)
    alpha     = 0.05
    test_lags = [3, 4, 5]

    def run_test(ret):
        var_y  = np.var(ret, ddof=1)
        se     = var_y / np.sqrt(len(ret))
        p_list = []
        for m in test_lags:
            lagm  = lagm_cov_est(ret, m)
            z_m   = lagm / se
            p_val = 2*(1 - norm.cdf(abs(z_m)))
            p_list.append(p_val)
        bon_p = min(min(p_list)*len(p_list), 1.0)
        return bon_p

    # ── Step 1: Size under H0 = BS ─────────────────────────────────────────
    print("Step 1 — Test size under H0 = Black-Scholes …")
    size_rej = 0
    size_n   = 500
    sigma    = np.sqrt(TRUE_heston['theta'])
    mu       = TRUE_heston['mu']

    for s in range(size_n):
        ret_bs = np.random.normal(
            (mu - 0.5*sigma**2)*h, sigma*np.sqrt(h), 1000
        )
        bon_p = run_test(ret_bs)
        if bon_p < alpha:
            size_rej += 1

    print(f"  Test size (BS, N=1000): {size_rej/size_n*100:.1f}%  (target ~5%)\n")

    # ── Step 2: Size under H0 = Heston ────────────────────────────────────
    # Note: Heston HAS nonzero autocovariance, so the test SHOULD reject
    # under Heston. This test distinguishes Heston FROM BS, not as a GoF.
    print("Step 2 — Rejection rate under Heston (should be HIGH) …")
    heston_rej = 0
    heston_n   = 200

    for s in range(heston_n):
        ret_h = heston_returns(
            float(TRUE_heston['theta']), int(10_000),
            float(TRUE_heston['mu']), float(TRUE_heston['theta']),
            float(TRUE_heston['kappa']), float(TRUE_heston['sigma_V']),
            float(TRUE_heston['rho']), float(h), n_sub
        )
        bon_p = run_test(ret_h)
        if bon_p < alpha:
            heston_rej += 1

    print(f"  Rejection rate (Heston, N=10k): "
          f"{heston_rej/heston_n*100:.1f}%  (higher = more power)\n")

    # ── Step 3: Power curve — BS data, test whether lags nonzero ──────────
    print("Step 3 — Power curve (H0: BS, H1: Heston)")
    print(f"{'N':>8s}  {'Power':>8s}  {'Yrs monthly':>14s}  "
          f"{'Yrs weekly':>12s}  {'Valid':>7s}")
    print("-" * 56)

    results = []
    for N in N_vals:
        rejections = 0
        valid      = 0

        for s in range(n_sim):
            # Simulate from HESTON — test whether lags are detectable
            ret_h = heston_returns(
                float(TRUE_heston['theta']), int(N),
                float(TRUE_heston['mu']), float(TRUE_heston['theta']),
                float(TRUE_heston['kappa']), float(TRUE_heston['sigma_V']),
                float(TRUE_heston['rho']), float(h), n_sub
            )
            bon_p = run_test(ret_h)
            if np.isfinite(bon_p):
                if bon_p < alpha:
                    rejections += 1
                valid += 1

        power = rejections/valid if valid > 0 else np.nan
        results.append({'N': N, 'power': power})
        print(f"  {N:>6,}  {power:>7.1%}  {N/12:>12.1f} yrs  "
              f"{N/52:>10.1f} yrs  {valid:>7d}")

    # ── Step 4: Minimum N for 80% power ───────────────────────────────────
    powers = [r['power'] for r in results]
    n_80   = None
    for i in range(len(powers)-1):
        if (np.isfinite(powers[i]) and np.isfinite(powers[i+1]) and
                powers[i] < 0.8 <= powers[i+1]):
            frac = (0.8-powers[i])/(powers[i+1]-powers[i])
            n_80 = int(N_vals[i] + frac*(N_vals[i+1]-N_vals[i]))
            break

    print(f"\nMinimum N to detect Heston autocorrelation at 80% power:")
    if n_80:
        print(f"  N ≈ {n_80:,} observations")
        print(f"  Monthly (h=21/252) : ~{n_80//12:,} years")
        print(f"  Weekly  (h=5/252)  : ~{n_80//52:,} years")
        print(f"  SPY (N=227 monthly): estimated power ≈ "
              f"{float(np.interp(227, N_vals, powers)):.1%}")
    else:
        max_p = max(p for p in powers if np.isfinite(p))
        print(f"  80% power not reached. Max: {max_p:.1%}")

    return results, n_80


# ── Run ────────────────────────────────────────────────────────────────────────
TRUE = dict(mu=0.125, kappa=0.1, theta=0.25, sigma_V=0.1, rho=-0.7)
h    = 1.0

results, n_80 = power_analysis_lag_test(
    TRUE_heston = TRUE,
    h           = h,
    n_sim       = 500,
    N_vals      = [50, 100, 200, 500, 1000, 2000, 5000, 10000, 50000, 100000]
)

# ── Apply to real data ─────────────────────────────────────────────────────────
print("\n" + "="*55)
print("Real data — testing for stochastic volatility signal")
print("="*55)

for ticker, ret in [('SPY', ret_spy), ('GLD', ret_gld), ('TLT', ret_tlt)]:
    print(f"\n{ticker} (N={len(ret):,} monthly returns):")
    lag_autocov_test(ret, h=21/252, test_lags=[3, 4, 5])

: 

In [ ]:
def interpret_results(ticker, p_vals, N, power_at_N):
    """
    Prints a concise interpretation of the GoF test results
    accounting for test power.
    """
    from scipy.stats import norm

    bon_p   = min(min(p for p in p_vals if np.isfinite(p))*3, 1.0)
    rejects = bon_p < 0.05

    print(f"\n{ticker} — Interpretation")
    print(f"  N = {N:,} monthly returns  (test power ≈ {power_at_N:.1%})")
    print(f"  Bonferroni p = {bon_p:.4f}  "
          f"({'reject H0' if rejects else 'cannot reject H0'})")

    if not rejects:
        print(f"\n  The test does not reject the null of zero autocovariance.")
        print(f"  However, with only {N:,} observations the test has only "
              f"~{power_at_N:.0%} power.")
        print(f"  This means even if Heston stochastic volatility were present,")
        print(f"  the test would fail to detect it ~{1-power_at_N:.0%} of the time.")
        print(f"  Conclusion: the result is UNINFORMATIVE, not evidence against")
        print(f"  stochastic volatility.")
    else:
        print(f"\n  The test rejects zero autocovariance — evidence of")
        print(f"  stochastic volatility structure in monthly returns.")


# Interpolate power at N=227 from the power curve
power_227 = float(np.interp(227,
                             [r['N'] for r in results],
                             [r['power'] for r in results]))

for ticker, p_vals_real in [
    ('SPY', lag_autocov_test(ret_spy, 21/252)),
    ('GLD', lag_autocov_test(ret_gld, 21/252)),
    ('TLT', lag_autocov_test(ret_tlt, 21/252)),
]:
    interpret_results(ticker, p_vals_real, 227, power_227)

: 

In [ ]:
from scipy.stats import norm
import matplotlib.pyplot as plt
import numpy as np

N_vals   = [50, 100, 200, 500, 1000, 2000, 5000, 10000, 50000, 100000]
N_actual = 227
colors   = {'SPY': '#1565C0', 'GLD': '#2E7D32', 'TLT': '#AD1457'}

fig, ax = plt.subplots(figsize=(10, 6))

for ticker in ['SPY', 'GLD', 'TLT']:
    res    = asset_results[ticker]
    powers = res['powers']
    valid  = [np.isfinite(p) for p in powers]
    N_plot = [N for N, v in zip(N_vals, valid) if v]
    P_plot = [p for p, v in zip(powers, valid) if v]

    ax.plot(N_plot, P_plot, 'o-',
            color=colors[ticker], lw=2, ms=5, label=ticker)

    # Mark power at actual sample size
    p_at_N = float(np.interp(N_actual, N_vals, powers))
    ax.scatter(N_actual, p_at_N,
               color=colors[ticker], s=150, zorder=6,
               marker='*', label=f'{ticker} at N={N_actual}: {p_at_N:.1%}')

ax.axhline(0.80, color='red',   lw=1.5, ls='--', label='80% power target')
ax.axhline(0.05, color='grey',  lw=1.0, ls=':',  label='5% nominal size')
ax.axvline(N_actual, color='black', lw=1.5, ls='--',
           label=f'N={N_actual} (our sample)', alpha=0.7)

ax.set_xscale('log')
ax.set_ylim(-0.02, 1.05)
ax.set_xlabel('Sample size N (log scale)', fontsize=12)
ax.set_ylabel('Power (rejection rate)', fontsize=12)
#ax.set_title('Power of Autocovariance Test for Stochastic Volatility\n'
#             'SPY, GLD and TLT — Monthly Returns (h = 21/252)\n'
#             '(★ marks estimated power at actual sample size N = 227)',
#             fontsize=11, fontweight='bold')
ax.legend(fontsize=9, ncol=2)
ax.grid(True, alpha=0.3, ls=':')

# Secondary x-axis — years of monthly data
ax2 = ax.twiny()
ax2.set_xscale('log')
ax2.set_xlim(ax.get_xlim())
tick_N    = [100, 500, 1000, 5000, 10000, 50000, 100000]
tick_labs = [f'{int(N/12)}' for N in tick_N]
ax2.set_xticks(tick_N)
ax2.set_xticklabels(tick_labs, fontsize=9)
ax2.set_xlabel('Years of monthly data', fontsize=10)

plt.tight_layout()
plt.savefig('power_curves_assets.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved to power_curves_assets.png")

: 

In [ ]:
asset_results = {}

for ticker, est in asset_est.items():
    print(f"  {ticker} …")
    powers = power_curve_per_asset(est, h=h_m, n_sim=300, N_vals=N_vals)
    asset_results[ticker] = {'powers': powers, 'est': est}

: 

In [ ]:
print(f"\n{'Ticker':<8s}  {'κ':>8s}  {'θ':>7s}  {'σ_V':>6s}  "
      f"{'Power@227':>10s}  {'N for 80%':>12s}  {'Yrs monthly':>12s}")
print("-" * 68)

for ticker in ['SPY', 'GLD', 'TLT']:
    est    = asset_results[ticker]['est']
    powers = asset_results[ticker]['powers']
    p_227  = float(np.interp(N_actual, N_vals, powers))

    n_80 = None
    for i in range(len(powers)-1):
        if (np.isfinite(powers[i]) and np.isfinite(powers[i+1]) and
                powers[i] < 0.8 <= powers[i+1]):
            frac = (0.8-powers[i])/(powers[i+1]-powers[i])
            n_80 = int(N_vals[i] + frac*(N_vals[i+1]-N_vals[i]))
            break

    n80_str = f"{n_80:>12,}"       if n_80 else f"{'> 100,000':>12s}"
    yrs_str = f"{n_80//12:>10,} yrs" if n_80 else f"{'> 8,333':>10s} yrs"

    print(f"  {ticker:<6s}  {est['kappa']:>8.3f}  {est['theta']:>7.4f}  "
          f"{est['sigma_V']:>6.3f}  {p_227:>9.1%}  {n80_str}  {yrs_str}")

: 